<a href="https://colab.research.google.com/github/Eddycrack864/UVR5-NO-UI/blob/main/UVR5_NO_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🎵 UVR5 NO UI 🎵**

Colab created by **[Not Eddy (Spanish Mod)](http://discord.com/users/274566299349155851)** in **[AI HUB](https://discord.gg/aihub)** community.

<br>**This is a CLI type colab (the no-UI version of [UVR5-UI](https://github.com/Eddycrack864/UVR5-UI)) based on the [python-audio-separator](https://github.com/nomadkaraoke/python-audio-separator) repository.<br>
If you liked this colab you can star my repo on [GitHub](https://github.com/Eddycrack864/UVR5-NO-UI)**.<br>

<br>You can donate to the original UVR5 project here:<br>
[!["Buy Me A Coffee"](https://www.buymeacoffee.com/assets/img/custom_images/orange_img.png)](https://www.buymeacoffee.com/uvr5)

<br>**How To Use: https://github.com/Eddycrack864/UVR5-NO-UI#how-to-use**

![](https://count.nett.moe/get/uvr5_no_ui_colab/img?theme=rule34)

In [ ]:
#@title # **Installation**
#@markdown ####This takes around 2 minutes. Run it once per session, before any other cell.
#@markdown Mount Google Drive (Colab only) to read and save your audio files from/to your Drive:
mount_drive = True #@param {type:"boolean"}

import os
from IPython.display import clear_output

if os.path.exists("/content"):
    ENV, path = "colab", "/content"
elif os.path.exists("/kaggle/working"):
    ENV, path = "kaggle", "/kaggle/working"
else:
    ENV, path = "local", os.getcwd()

if ENV == "colab" and mount_drive:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

!pip install -q uv
!uv pip install --system -q "audio-separator[gpu]==0.47.0" "librosa<1.0" "yt_dlp[default]" demucs
!uv pip install --system -q "onnxruntime-gpu[cuda]==1.26.0"
# JavaScript runtime that yt-dlp needs for YouTube links
!uv pip install --system -q deno

# ======================================================================
# Shared helpers used by every cell below
# ======================================================================
import gc
import json
import logging
import math
import re
import shutil
import string
import tempfile
import time
import urllib.request
from pathlib import Path

MODELS_DIR = Path(path) / "models"
TEMP_DIR = Path(path) / "temp"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FORMATS = ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]
AUDIO_EXTENSIONS = tuple(f".{ext}" for ext in OUTPUT_FORMATS)
DEFAULT_NAME_TEMPLATE = "{input}_({stem})_{model}"
ARCH_LABELS = {"roformer": "Roformer", "mdx23c": "MDX23C", "mdxnet": "MDX-NET", "vrarch": "VR Arch", "demucs": "Demucs"}

# >>> MODELS (generated by tools/sync_models.py from UVR5-UI/assets/models.json, do not edit by hand)
MODELS = {
    "roformer": {
        "BS-Roformer-Viperx-1297": "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
        "BS-Roformer-Viperx-1296": "model_bs_roformer_ep_368_sdr_12.9628.ckpt",
        "BS-Roformer-Viperx-1053": "model_bs_roformer_ep_937_sdr_10.5309.ckpt",
        "Mel-Roformer-Viperx-1143": "model_mel_band_roformer_ep_3005_sdr_11.4360.ckpt",
        "BS-Roformer-De-Reverb": "deverb_bs_roformer_8_384dim_10depth.ckpt",
        "Mel-Roformer-Crowd-Aufr33-Viperx": "mel_band_roformer_crowd_aufr33_viperx_sdr_8.7144.ckpt",
        "Mel-Roformer-Denoise-Aufr33": "denoise_mel_band_roformer_aufr33_sdr_27.9959.ckpt",
        "Mel-Roformer-Denoise-Aufr33-Aggr": "denoise_mel_band_roformer_aufr33_aggr_sdr_27.9768.ckpt",
        "MelBand Roformer | Denoise-Debleed by Gabox": "mel_band_roformer_denoise_debleed_gabox.ckpt",
        "Mel-Roformer-Karaoke-Aufr33-Viperx": "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
        "MelBand Roformer | Karaoke by Gabox": "mel_band_roformer_karaoke_gabox.ckpt",
        "MelBand Roformer | Karaoke V2 by Gabox": "mel_band_roformer_karaoke_gabox_v2.ckpt",
        "MelBand Roformer | Karaoke by becruily": "mel_band_roformer_karaoke_becruily.ckpt",
        "BS Roformer | Karaoke by frazer-becruily": "bs_roformer_karaoke_frazer_becruily.ckpt",
        "BS Roformer | Karaoke by anvuew": "bs_roformer_karaoke_anvuew.ckpt",
        "MelBand Roformer | Vocals by Kimberley Jensen": "vocals_mel_band_roformer.ckpt",
        "MelBand Roformer Kim | FT by unwa": "mel_band_roformer_kim_ft_unwa.ckpt",
        "MelBand Roformer Kim | FT 2 by unwa": "mel_band_roformer_kim_ft2_unwa.ckpt",
        "MelBand Roformer Kim | FT 2 Bleedless by unwa": "mel_band_roformer_kim_ft2_bleedless_unwa.ckpt",
        "MelBand Roformer Kim | FT 3 by unwa": "mel_band_roformer_kim_ft3_unwa.ckpt",
        "MelBand Roformer Kim | Inst V1 by Unwa": "melband_roformer_inst_v1.ckpt",
        "MelBand Roformer Kim | Inst V1 Plus by Unwa": "melband_roformer_inst_v1_plus.ckpt",
        "MelBand Roformer Kim | Inst V1 (E) by Unwa": "melband_roformer_inst_v1e.ckpt",
        "MelBand Roformer Kim | Inst V1 (E) Plus by Unwa": "melband_roformer_inst_v1e_plus.ckpt",
        "MelBand Roformer Kim | Inst V2 by Unwa": "melband_roformer_inst_v2.ckpt",
        "MelBand Roformer Kim | InstVoc Duality V1 by Unwa": "melband_roformer_instvoc_duality_v1.ckpt",
        "MelBand Roformer Kim | InstVoc Duality V2 by Unwa": "melband_roformer_instvox_duality_v2.ckpt",
        "MelBand Roformer | Vocals by becruily": "mel_band_roformer_vocals_becruily.ckpt",
        "MelBand Roformer | Instrumental by becruily": "mel_band_roformer_instrumental_becruily.ckpt",
        "MelBand Roformer | Vocals Fullness by Aname": "mel_band_roformer_vocal_fullness_aname.ckpt",
        "BS Roformer | Vocals by Gabox": "bs_roformer_vocals_gabox.ckpt",
        "MelBand Roformer | Vocals by Gabox": "mel_band_roformer_vocals_gabox.ckpt",
        "MelBand Roformer | Vocals V2 by Gabox": "mel_band_roformer_vocals_v2_gabox.ckpt",
        "MelBand Roformer | Vocals FV1 by Gabox": "mel_band_roformer_vocals_fv1_gabox.ckpt",
        "MelBand Roformer | Vocals FV2 by Gabox": "mel_band_roformer_vocals_fv2_gabox.ckpt",
        "MelBand Roformer | Vocals FV3 by Gabox": "mel_band_roformer_vocals_fv3_gabox.ckpt",
        "MelBand Roformer | Vocals FV4 by Gabox": "mel_band_roformer_vocals_fv4_gabox.ckpt",
        "MelBand Roformer | Vocals FV5 by Gabox": "mel_band_roformer_vocals_fv5_gabox.ckpt",
        "MelBand Roformer | Vocals FV6 by Gabox": "mel_band_roformer_vocals_fv6_gabox.ckpt",
        "MelBand Roformer | Vocals FV7b by Gabox": "mel_band_roformer_vocals_fv7b_gabox.ckpt",
        "MelBand Roformer | Instrumental by Gabox": "mel_band_roformer_instrumental_gabox.ckpt",
        "MelBand Roformer | Instrumental 2 by Gabox": "mel_band_roformer_instrumental_2_gabox.ckpt",
        "MelBand Roformer | Instrumental 3 by Gabox": "mel_band_roformer_instrumental_3_gabox.ckpt",
        "MelBand Roformer | Instrumental Bleedless V1 by Gabox": "mel_band_roformer_instrumental_bleedless_v1_gabox.ckpt",
        "MelBand Roformer | Instrumental Bleedless V2 by Gabox": "mel_band_roformer_instrumental_bleedless_v2_gabox.ckpt",
        "MelBand Roformer | Instrumental Bleedless V3 by Gabox": "mel_band_roformer_instrumental_bleedless_v3_gabox.ckpt",
        "MelBand Roformer | Instrumental Fullness V1 by Gabox": "mel_band_roformer_instrumental_fullness_v1_gabox.ckpt",
        "MelBand Roformer | Instrumental Fullness V2 by Gabox": "mel_band_roformer_instrumental_fullness_v2_gabox.ckpt",
        "MelBand Roformer | Instrumental Fullness V3 by Gabox": "mel_band_roformer_instrumental_fullness_v3_gabox.ckpt",
        "MelBand Roformer | Instrumental Fullness V4 by Gabox": "mel_band_roformer_instrumental_fullness_v4_gabox.ckpt",
        "MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox": "mel_band_roformer_instrumental_fullness_noise_v4_gabox.ckpt",
        "MelBand Roformer | INSTV5 by Gabox": "mel_band_roformer_instrumental_instv5_gabox.ckpt",
        "MelBand Roformer | INSTV5N by Gabox": "mel_band_roformer_instrumental_instv5n_gabox.ckpt",
        "MelBand Roformer | INSTV6 by Gabox": "mel_band_roformer_instrumental_instv6_gabox.ckpt",
        "MelBand Roformer | INSTV6N by Gabox": "mel_band_roformer_instrumental_instv6n_gabox.ckpt",
        "MelBand Roformer | INSTV7 by Gabox": "mel_band_roformer_instrumental_instv7_gabox.ckpt",
        "MelBand Roformer | INSTV7N by Gabox": "mel_band_roformer_instrumental_instv7n_gabox.ckpt",
        "MelBand Roformer | INSTV8 by Gabox": "mel_band_roformer_instrumental_instv8_gabox.ckpt",
        "MelBand Roformer | INSTV8N by Gabox": "mel_band_roformer_instrumental_instv8n_gabox.ckpt",
        "MelBand Roformer | Instrumental FV7z by Gabox": "mel_band_roformer_instrumental_fv7z_gabox.ckpt",
        "MelBand Roformer | Instrumental FV8 by Gabox": "mel_band_roformer_instrumental_fv8_gabox.ckpt",
        "MelBand Roformer | Instrumental FV8b by Gabox": "mel_band_roformer_instrumental_fv8b_gabox.ckpt",
        "MelBand Roformer | Instrumental FVX by Gabox": "mel_band_roformer_instrumental_fvx_gabox.ckpt",
        "MelBand Roformer | De-Reverb by anvuew": "dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt",
        "MelBand Roformer | De-Reverb Less Aggressive by anvuew": "dereverb_mel_band_roformer_less_aggressive_anvuew_sdr_18.8050.ckpt",
        "MelBand Roformer | De-Reverb Mono by anvuew": "dereverb_mel_band_roformer_mono_anvuew.ckpt",
        "MelBand Roformer | De-Reverb Big by Sucial": "dereverb_big_mbr_ep_362.ckpt",
        "MelBand Roformer | De-Reverb Super Big by Sucial": "dereverb_super_big_mbr_ep_346.ckpt",
        "MelBand Roformer | De-Reverb-Echo by Sucial": "dereverb-echo_mel_band_roformer_sdr_10.0169.ckpt",
        "MelBand Roformer | De-Reverb-Echo V2 by Sucial": "dereverb-echo_mel_band_roformer_sdr_13.4843_v2.ckpt",
        "MelBand Roformer | De-Reverb-Echo Fused by Sucial": "dereverb_echo_mbr_fused.ckpt",
        "MelBand Roformer Kim | SYHFT by SYH99999": "MelBandRoformerSYHFT.ckpt",
        "MelBand Roformer Kim | SYHFT V2 by SYH99999": "MelBandRoformerSYHFTV2.ckpt",
        "MelBand Roformer Kim | SYHFT V2.5 by SYH99999": "MelBandRoformerSYHFTV2.5.ckpt",
        "MelBand Roformer Kim | SYHFT V3 by SYH99999": "MelBandRoformerSYHFTV3Epsilon.ckpt",
        "MelBand Roformer Kim | Big SYHFT V1 by SYH99999": "MelBandRoformerBigSYHFTV1.ckpt",
        "MelBand Roformer Kim | Big Beta 4 FT by unwa": "melband_roformer_big_beta4.ckpt",
        "MelBand Roformer Kim | Big Beta 5e FT by unwa": "melband_roformer_big_beta5e.ckpt",
        "MelBand Roformer | Big Beta 6 by unwa": "melband_roformer_big_beta6.ckpt",
        "MelBand Roformer | Big Beta 6X by unwa": "melband_roformer_big_beta6x.ckpt",
        "BS Roformer | Vocals Revive by Unwa": "bs_roformer_vocals_revive_unwa.ckpt",
        "BS Roformer | Vocals Revive V2 by Unwa": "bs_roformer_vocals_revive_v2_unwa.ckpt",
        "BS Roformer | Vocals Revive V3e by Unwa": "bs_roformer_vocals_revive_v3e_unwa.ckpt",
        "BS Roformer | Chorus Male-Female by Sucial": "model_chorus_bs_roformer_ep_267_sdr_24.1275.ckpt",
        "BS Roformer | Male-Female by aufr33": "bs_roformer_male_female_by_aufr33_sdr_7.2889.ckpt",
        "MelBand Roformer | Aspiration by Sucial": "aspiration_mel_band_roformer_sdr_18.9845.ckpt",
        "MelBand Roformer | Aspiration Less Aggressive by Sucial": "aspiration_mel_band_roformer_less_aggr_sdr_18.1201.ckpt",
        "MelBand Roformer | Bleed Suppressor V1 by unwa-97chris": "mel_band_roformer_bleed_suppressor_v1.ckpt",
        "BS Roformer | Vocals Resurrection by unwa": "bs_roformer_vocals_resurrection_unwa.ckpt",
        "BS Roformer | Instrumental Resurrection by unwa": "bs_roformer_instrumental_resurrection_unwa.ckpt",
        "BS Roformer | Instrumental ResurrectioN by Gabox": "bs_roformer_instrumental_resurrection_gabox.ckpt",
        "BS Roformer SW by jarredou": "BS-Roformer-SW.ckpt",
    },
    "mdx23c": {
        "MDX23C_D1581.ckpt": "MDX23C_D1581.ckpt",
        "MDX23C-8KFFT-InstVoc_HQ.ckpt": "MDX23C-8KFFT-InstVoc_HQ.ckpt",
        "MDX23C-8KFFT-InstVoc_HQ_2.ckpt": "MDX23C-8KFFT-InstVoc_HQ_2.ckpt",
        "MDX23C-De-Reverb-aufr33-jarredou.ckpt": "MDX23C-De-Reverb-aufr33-jarredou.ckpt",
        "MDX23C-DrumSep-aufr33-jarredou.ckpt": "MDX23C-DrumSep-aufr33-jarredou.ckpt",
    },
    "mdxnet": {
        "UVR-MDX-NET-Inst_full_292.onnx": "UVR-MDX-NET-Inst_full_292.onnx",
        "UVR-MDX-NET_Inst_187_beta.onnx": "UVR-MDX-NET_Inst_187_beta.onnx",
        "UVR-MDX-NET_Inst_82_beta.onnx": "UVR-MDX-NET_Inst_82_beta.onnx",
        "UVR-MDX-NET_Inst_90_beta.onnx": "UVR-MDX-NET_Inst_90_beta.onnx",
        "UVR-MDX-NET_Main_340.onnx": "UVR-MDX-NET_Main_340.onnx",
        "UVR-MDX-NET_Main_390.onnx": "UVR-MDX-NET_Main_390.onnx",
        "UVR-MDX-NET_Main_406.onnx": "UVR-MDX-NET_Main_406.onnx",
        "UVR-MDX-NET_Main_427.onnx": "UVR-MDX-NET_Main_427.onnx",
        "UVR-MDX-NET_Main_438.onnx": "UVR-MDX-NET_Main_438.onnx",
        "UVR-MDX-NET-Inst_HQ_1.onnx": "UVR-MDX-NET-Inst_HQ_1.onnx",
        "UVR-MDX-NET-Inst_HQ_2.onnx": "UVR-MDX-NET-Inst_HQ_2.onnx",
        "UVR-MDX-NET-Inst_HQ_3.onnx": "UVR-MDX-NET-Inst_HQ_3.onnx",
        "UVR-MDX-NET-Inst_HQ_4.onnx": "UVR-MDX-NET-Inst_HQ_4.onnx",
        "UVR-MDX-NET-Inst_HQ_5.onnx": "UVR-MDX-NET-Inst_HQ_5.onnx",
        "UVR_MDXNET_Main.onnx": "UVR_MDXNET_Main.onnx",
        "UVR-MDX-NET-Inst_Main.onnx": "UVR-MDX-NET-Inst_Main.onnx",
        "UVR_MDXNET_1_9703.onnx": "UVR_MDXNET_1_9703.onnx",
        "UVR_MDXNET_2_9682.onnx": "UVR_MDXNET_2_9682.onnx",
        "UVR_MDXNET_3_9662.onnx": "UVR_MDXNET_3_9662.onnx",
        "UVR-MDX-NET-Inst_1.onnx": "UVR-MDX-NET-Inst_1.onnx",
        "UVR-MDX-NET-Inst_2.onnx": "UVR-MDX-NET-Inst_2.onnx",
        "UVR-MDX-NET-Inst_3.onnx": "UVR-MDX-NET-Inst_3.onnx",
        "UVR_MDXNET_KARA.onnx": "UVR_MDXNET_KARA.onnx",
        "UVR_MDXNET_KARA_2.onnx": "UVR_MDXNET_KARA_2.onnx",
        "UVR_MDXNET_9482.onnx": "UVR_MDXNET_9482.onnx",
        "UVR-MDX-NET-Voc_FT.onnx": "UVR-MDX-NET-Voc_FT.onnx",
        "Kim_Vocal_1.onnx": "Kim_Vocal_1.onnx",
        "Kim_Vocal_2.onnx": "Kim_Vocal_2.onnx",
        "Kim_Inst.onnx": "Kim_Inst.onnx",
        "Reverb_HQ_By_FoxJoy.onnx": "Reverb_HQ_By_FoxJoy.onnx",
        "UVR-MDX-NET_Crowd_HQ_1.onnx": "UVR-MDX-NET_Crowd_HQ_1.onnx",
        "kuielab_a_vocals.onnx": "kuielab_a_vocals.onnx",
        "kuielab_a_other.onnx": "kuielab_a_other.onnx",
        "kuielab_a_bass.onnx": "kuielab_a_bass.onnx",
        "kuielab_a_drums.onnx": "kuielab_a_drums.onnx",
        "kuielab_b_vocals.onnx": "kuielab_b_vocals.onnx",
        "kuielab_b_other.onnx": "kuielab_b_other.onnx",
        "kuielab_b_bass.onnx": "kuielab_b_bass.onnx",
        "kuielab_b_drums.onnx": "kuielab_b_drums.onnx",
    },
    "vrarch": {
        "1_HP-UVR.pth": "1_HP-UVR.pth",
        "2_HP-UVR.pth": "2_HP-UVR.pth",
        "3_HP-Vocal-UVR.pth": "3_HP-Vocal-UVR.pth",
        "4_HP-Vocal-UVR.pth": "4_HP-Vocal-UVR.pth",
        "5_HP-Karaoke-UVR.pth": "5_HP-Karaoke-UVR.pth",
        "6_HP-Karaoke-UVR.pth": "6_HP-Karaoke-UVR.pth",
        "7_HP2-UVR.pth": "7_HP2-UVR.pth",
        "8_HP2-UVR.pth": "8_HP2-UVR.pth",
        "9_HP2-UVR.pth": "9_HP2-UVR.pth",
        "10_SP-UVR-2B-32000-1.pth": "10_SP-UVR-2B-32000-1.pth",
        "11_SP-UVR-2B-32000-2.pth": "11_SP-UVR-2B-32000-2.pth",
        "12_SP-UVR-3B-44100.pth": "12_SP-UVR-3B-44100.pth",
        "13_SP-UVR-4B-44100-1.pth": "13_SP-UVR-4B-44100-1.pth",
        "14_SP-UVR-4B-44100-2.pth": "14_SP-UVR-4B-44100-2.pth",
        "15_SP-UVR-MID-44100-1.pth": "15_SP-UVR-MID-44100-1.pth",
        "16_SP-UVR-MID-44100-2.pth": "16_SP-UVR-MID-44100-2.pth",
        "17_HP-Wind_Inst-UVR.pth": "17_HP-Wind_Inst-UVR.pth",
        "UVR-De-Echo-Aggressive.pth": "UVR-De-Echo-Aggressive.pth",
        "UVR-De-Echo-Normal.pth": "UVR-De-Echo-Normal.pth",
        "UVR-DeEcho-DeReverb.pth": "UVR-DeEcho-DeReverb.pth",
        "UVR-De-Reverb-aufr33-jarredou.pth": "UVR-De-Reverb-aufr33-jarredou.pth",
        "UVR-DeNoise-Lite.pth": "UVR-DeNoise-Lite.pth",
        "UVR-DeNoise.pth": "UVR-DeNoise.pth",
        "UVR-BVE-4B_SN-44100-1.pth": "UVR-BVE-4B_SN-44100-1.pth",
        "UVR-BVE-4B_SN-44100-2.pth": "UVR-BVE-4B_SN-44100-2.pth",
        "MGM_HIGHEND_v4.pth": "MGM_HIGHEND_v4.pth",
        "MGM_LOWEND_A_v4.pth": "MGM_LOWEND_A_v4.pth",
        "MGM_LOWEND_B_v4.pth": "MGM_LOWEND_B_v4.pth",
        "MGM_MAIN_v4.pth": "MGM_MAIN_v4.pth",
    },
    "demucs": {
        "htdemucs_ft.yaml": "htdemucs_ft.yaml",
        "htdemucs_6s.yaml": "htdemucs_6s.yaml",
        "htdemucs.yaml": "htdemucs.yaml",
        "hdemucs_mmi.yaml": "hdemucs_mmi.yaml",
    },
}
RECOMMENDED = {"Instrumental": "BS Roformer | Instrumental Resurrection by unwa", "Vocals": "BS Roformer SW by jarredou", "Karaoke": "BS Roformer | Karaoke by anvuew", "De-Reverb": "MelBand Roformer | De-Reverb by anvuew", "Denoise": "MelBand Roformer | Denoise-Debleed by Gabox"}
# <<< MODELS

DRUMSEP_URL = "https://huggingface.co/Eddycrack864/Drumsep/resolve/main/modelo_final.th"

# Default parameters of each architecture (same as UVR5-UI). The ensemble uses the
# parameters of the last separation run with each architecture, or these.
ARCH_DEFAULTS = {
    "roformer": dict(segment_size=256, override_model_segment_size=False, overlap=8, use_model_overlap=False,
                     batch_size=1, pitch_shift=0, normalization_threshold=0.9, amplification_threshold=0.7),
    "mdx23c": dict(segment_size=256, override_model_segment_size=False, overlap=8, use_model_overlap=False,
                   batch_size=1, pitch_shift=0, normalization_threshold=0.9, amplification_threshold=0.7),
    "mdxnet": dict(hop_length=1024, segment_size=256, overlap=0.25, batch_size=1, enable_denoise=True,
                   invert_using_spec=False, normalization_threshold=0.9, amplification_threshold=0.7),
    "vrarch": dict(window_size=512, aggression=5, batch_size=1, enable_tta=True, enable_post_process=False,
                   post_process_threshold=0.2, high_end_process=False, normalization_threshold=0.9,
                   amplification_threshold=0.7),
    "demucs": dict(shifts=2, segment_size=40, segments_enabled=True, overlap=0.25, normalization_threshold=0.9,
                   amplification_threshold=0.7),
}
LAST_PARAMS = {}

# Global settings, changed by the "Settings" cell
PRECISIONS = ("auto", "fp32", "autocast", "native_fp16")
SETTINGS = {"precision": "auto", "torch_compile": False, "chunk_duration": 0, "use_soundfile": False,
            "name_template": DEFAULT_NAME_TEMPLATE}


class UserError(Exception):
    """A problem the user can fix (wrong path, bad option...). Printed without a traceback."""


def print_error(e):
    print(f"\n❌ {e}")


# ---------- paths ----------
def clean_path(value):
    """Strip whitespace and the quotes that "Copy path" adds around a path."""
    value = (value or "").strip()
    if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
        value = value[1:-1].strip()
    return value


def is_url(value):
    return clean_path(value).lower().startswith(("http://", "https://"))


def drive_mounted():
    return os.path.isdir("/content/drive/MyDrive")


def default_output_dir():
    return Path("/content/drive/MyDrive/UVR5/outputs") if drive_mounted() else Path(path) / "outputs"


def list_audio_files(folder):
    """Audio files directly inside ``folder`` (not recursive), sorted by name."""
    return sorted((p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS),
                  key=lambda p: p.name.lower())


def resolve_output(output_folder):
    value = clean_path(output_folder)
    unavailable = value.startswith("/content/drive") and not drive_mounted()
    if not value or unavailable or (ENV != "colab" and value.startswith("/content")):
        fallback = default_output_dir()
        if value:
            print(f"⚠️ Output folder {value} is not available here, using {fallback}")
        value = str(fallback)
    output_dir = Path(value)
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
    except OSError as e:
        raise UserError(f"Could not create the output folder {output_dir}: {e}") from None
    return output_dir


def resolve_inputs(audio_input):
    """Return (audio files, temporary folder to delete afterwards or None).

    ``audio_input`` can be an audio file, a folder (batch) or a link (downloaded with yt-dlp).
    """
    value = clean_path(audio_input)
    if not value or value == "Place the path or link here":
        raise UserError("Set audio_input to an audio file, a folder or a link.")
    if is_url(value):
        temp = Path(tempfile.mkdtemp(prefix="ytdl_", dir=TEMP_DIR))
        try:
            return [download_audio(value, temp)], temp
        except Exception:
            shutil.rmtree(temp, ignore_errors=True)
            raise
    p = Path(value)
    if p.is_file():
        if p.suffix.lower() not in AUDIO_EXTENSIONS:
            raise UserError(f"Unsupported audio file: {p.name}. Supported: {', '.join(AUDIO_EXTENSIONS)}")
        return [p], None
    if p.is_dir():
        return list_audio_files(p), None
    if value.startswith("/content/drive") and not drive_mounted():
        raise UserError("Google Drive is not mounted: run the Installation cell with mount_drive enabled.")
    raise UserError(f"Input path not found: {value}")


# ---------- yt-dlp ----------
YTDL_CODECS = {"ogg": "vorbis"}  # yt-dlp codec names that differ from the extension


def download_audio(url, output_dir, audio_format="wav", name=None):
    """Download the audio of ``url`` into ``output_dir`` and return the file path."""
    import yt_dlp

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    opts = {
        "format": "bestaudio/best",
        "noplaylist": True,
        "windowsfilenames": True,
        "outtmpl": str(output_dir / f"{name or '%(title)s'}.%(ext)s"),
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": YTDL_CODECS.get(audio_format, audio_format),
                            "preferredquality": "0"}],
    }
    if audio_format == "wav":
        opts["postprocessor_args"] = ["-acodec", "pcm_f32le"]
    print(f"Downloading audio from {url}...")
    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)
            files = [d.get("filepath") for d in info.get("requested_downloads") or [] if d.get("filepath")]
            fallback = Path(ydl.prepare_filename(info)).with_suffix(f".{audio_format}")
    except Exception as e:
        raise UserError(f"Could not download {url}: {e}") from None
    result = Path(files[0]) if files else fallback
    if not result.exists():
        raise UserError(f"The download finished but the audio file was not found: {result}")
    return result


# ---------- output names ----------
_OUTPUT_NAME_RE = re.compile(r"^(?P<input>.*)_\((?P<stem>[^()]+)\)_(?P<model>.*)$")
_STEM_RE = re.compile(r"_\(([^()]+)\)")
_UNSAFE_CHARS_RE = re.compile(r'[<>:"/\\|?*\x00-\x1f]')


def check_name_template(template):
    try:
        fields = [f for _, f, _, _ in string.Formatter().parse(template) if f is not None]
    except ValueError as e:
        raise UserError(f"Invalid name template: {e}") from None
    unknown = sorted({f for f in fields if f not in ("input", "stem", "model")})
    if unknown:
        raise UserError(f"Unknown placeholder in name template: {{{unknown[0]}}}. Use {{input}}, {{stem}} and {{model}}")
    if "stem" not in fields:
        raise UserError("The name template must include {stem}, or every stem would get the same name")


def render_name(template, input, stem, model):
    name = template.format(input=input, stem=stem, model=model)
    return _UNSAFE_CHARS_RE.sub("_", name).strip(" .") or f"{input}_{stem}"


def parse_stem_label(filename):
    matches = _STEM_RE.findall(Path(filename).stem)
    return matches[-1] if matches else None


def rename_output(file, template):
    """Rename an audio-separator output ``<input>_(<Stem>)_<model>.<ext>`` following ``template``."""
    file = Path(file)
    match = _OUTPUT_NAME_RE.match(file.stem)
    if template == DEFAULT_NAME_TEMPLATE or not match or not file.exists():
        return file
    target = file.with_name(render_name(template, **match.groupdict()) + file.suffix)
    if target != file:
        os.replace(file, target)
    return target


# ---------- audio I/O (ensemble and drumsep) ----------
SOUNDFILE_FORMATS = {"wav": "WAV", "flac": "FLAC", "ogg": "OGG", "aiff": "AIFF"}
PYDUB_FORMATS = {"mp3": "mp3", "m4a": "mp4", "opus": "opus", "ac3": "ac3"}


def load_audio(file, sample_rate):
    """Decode ``file`` as float32 (channels, samples) at ``sample_rate``."""
    import librosa
    import numpy as np

    wave, _ = librosa.load(str(file), sr=sample_rate, mono=False)
    return wave[np.newaxis, :] if wave.ndim == 1 else wave


def normalize_peak(wave, max_peak, min_peak=0.0):
    """Scale ``wave`` down to ``max_peak``, or up to ``min_peak`` when it is quieter (like audio-separator)."""
    import numpy as np

    peak = float(np.abs(wave).max()) if wave.size else 0.0
    if not np.isfinite(peak):
        raise ValueError("Audio data must contain only finite values")
    if peak > max_peak:
        return wave * (max_peak / peak)
    if min_peak and 0 < peak < min_peak:
        return wave * (min_peak / peak)
    return wave


def audio_subtype(file):
    import soundfile as sf

    try:
        return sf.info(str(file)).subtype
    except Exception:
        return None


def write_audio(file, wave, sample_rate, subtype=None):
    """Write ``wave`` (channels, samples) in the format given by the extension of ``file``."""
    import numpy as np

    file = Path(file)
    fmt = file.suffix.lower().lstrip(".")
    data = np.ascontiguousarray(np.asarray(wave, dtype=np.float32).T)
    file.parent.mkdir(parents=True, exist_ok=True)
    tmp = file.with_name(f".{file.stem}.part{file.suffix}")
    try:
        if fmt in SOUNDFILE_FORMATS:
            import soundfile as sf

            if subtype is not None and not sf.check_format(SOUNDFILE_FORMATS[fmt], subtype):
                subtype = None
            sf.write(str(tmp), data, sample_rate, format=SOUNDFILE_FORMATS[fmt], subtype=subtype)
        elif fmt in PYDUB_FORMATS:
            from pydub import AudioSegment

            pcm = (np.clip(data, -1.0, 1.0) * 32767).astype(np.int16)
            segment = AudioSegment(pcm.tobytes(), frame_rate=sample_rate, sample_width=2, channels=pcm.shape[1])
            segment.export(str(tmp), format=PYDUB_FORMATS[fmt], bitrate="320k" if fmt == "mp3" else None)
        else:
            raise ValueError(f"Unsupported output format: {fmt}")
        os.replace(tmp, file)
    finally:
        tmp.unlink(missing_ok=True)
    return file


# ---------- separator ----------
def describe_error(e):
    kind = type(e).__name__
    if kind == "AudioExportError":
        return f"could not write the output file ({e}). Check the output format"
    if kind == "InvalidAudioDataError":
        return f"the audio is empty or invalid ({e})"
    return str(e) or kind


def cuda_available():
    try:
        import torch

        return torch.cuda.is_available()
    except Exception:
        return False


def runtime_kwargs():
    """Separator kwargs of the global settings (precision, torch.compile, chunks, soundfile)."""
    precision = SETTINGS["precision"]
    cuda = cuda_available()
    if precision == "auto":
        precision = "autocast" if cuda else "fp32"
    elif precision in ("autocast", "native_fp16") and not cuda:
        print(f"⚠️ No GPU detected: {precision} is not available on CPU, using fp32")
        precision = "fp32"
    return {
        "use_autocast": precision == "autocast",
        "use_native_fp16": precision == "native_fp16",
        "use_torch_compile": bool(SETTINGS["torch_compile"]),
        "chunk_duration": float(SETTINGS["chunk_duration"]) or None,
        "use_soundfile": bool(SETTINGS["use_soundfile"]),
    }


def build_arch_kwargs(arch, params):
    """Turn the parameters of a separation cell into Separator kwargs."""
    p = dict(params)
    kwargs = {
        "normalization_threshold": float(p.pop("normalization_threshold", 0.9)),
        "amplification_threshold": float(p.pop("amplification_threshold", 0.7)),
        "output_single_stem": (p.pop("single_stem", "") or "").strip() or None,
    }
    if arch in ("roformer", "mdx23c"):
        if p.pop("use_model_overlap", False):
            p["overlap"] = None  # use the overlap of the model config
        kwargs["mdxc_params"] = p
    elif arch == "mdxnet":
        kwargs["invert_using_spec"] = bool(p.pop("invert_using_spec", False))
        kwargs["mdx_params"] = p
    elif arch == "vrarch":
        kwargs["vr_params"] = p
    elif arch == "demucs":
        kwargs["output_single_stem"] = None
        kwargs["demucs_params"] = p
    else:
        raise UserError(f"Unknown architecture: {arch}")
    return kwargs


# The last loaded Separator is kept so the next run with the same model and settings reuses it
_CACHE = {"key": None, "separator": None}


def release_model():
    _CACHE["key"] = _CACHE["separator"] = None
    gc.collect()
    if cuda_available():
        import torch

        torch.cuda.empty_cache()


def get_separator(model_filename, kwargs):
    """Return (Separator with ``model_filename`` loaded, loaded_now)."""
    key = (model_filename, json.dumps(kwargs, sort_keys=True, default=str))
    if _CACHE["key"] == key:
        return _CACHE["separator"], False
    release_model()  # free the (GPU) memory of the previous model first
    from audio_separator.separator import Separator

    separator = Separator(log_level=logging.WARNING, model_file_dir=str(MODELS_DIR), **kwargs)
    try:
        separator.load_model(model_filename=model_filename)
    except Exception as e:
        raise UserError(
            f"Could not load model {model_filename}: {describe_error(e)}\n"
            f"   If the model download was interrupted, delete {MODELS_DIR / model_filename} and run the cell again."
        ) from None
    _CACHE.update(key=key, separator=separator)
    return separator, True


def execution_summary(separator):
    precision = getattr(separator, "effective_precision", None)
    if not isinstance(precision, str):
        return ""
    return precision + (" + torch.compile" if getattr(separator, "effective_torch_compile", False) is True else "")


def format_time(seconds):
    minutes, seconds = divmod(int(seconds), 60)
    return f"{minutes}m {seconds}s" if minutes else f"{seconds}s"


def run_separation(arch, model, audio_input, output_folder, output_format, params):
    """Separate a file, every audio file of a folder (batch) or a link with one model."""
    try:
        _run_separation(arch, model, audio_input, output_folder, output_format, params)
    except UserError as e:
        print_error(e)


def _run_separation(arch, model, audio_input, output_folder, output_format, params):
    if model != "drumsep" and model not in MODELS.get(arch, {}):
        raise UserError(f"Unknown {ARCH_LABELS.get(arch, arch)} model: {model}")
    template = SETTINGS["name_template"]
    check_name_template(template)
    output_dir = resolve_output(output_folder)
    files, temp = resolve_inputs(audio_input)
    try:
        if not files:
            print("No valid audio files found.")
            return
        print(f"{len(files)} audio file(s) found")
        if model == "drumsep":
            _run_drumsep(files, output_dir, output_format, params, template)
            return

        LAST_PARAMS[arch] = {k: v for k, v in params.items() if k != "single_stem"}
        filename = MODELS[arch][model]
        kwargs = {"output_dir": str(output_dir), "output_format": output_format,
                  **runtime_kwargs(), **build_arch_kwargs(arch, params)}
        print(f"Loading model: {model}")
        separator, loaded = get_separator(filename, kwargs)
        summary = execution_summary(separator)
        print(("Model loaded" if loaded else "Reusing the loaded model") + (f" ({summary})" if summary else ""))

        start, failed = time.time(), []
        for i, file in enumerate(files, 1):
            print(f"\n[{i}/{len(files)}] Separating: {file.name}")
            try:
                outputs = separator.separate(str(file))
                if not outputs:
                    raise RuntimeError("no output produced")
                for name in outputs:
                    print(f"   ✔ {rename_output(output_dir / Path(name).name, template).name}")
            except Exception as e:
                failed.append(file.name)
                print(f"   ❌ Error separating {file.name}: {describe_error(e)}")
        print(f"\nDone: {len(files) - len(failed)} separated, {len(failed)} failed in "
              f"{format_time(time.time() - start)}. Output: {output_dir}")
    finally:
        if temp:
            shutil.rmtree(temp, ignore_errors=True)


# ---------- drumsep (Demucs model by inagoy) ----------
def _download_file(url, target):
    target = Path(target)
    if target.exists():
        return target
    print(f"Downloading {target.name}...")
    tmp = target.with_suffix(target.suffix + ".part")
    urllib.request.urlretrieve(url, tmp)
    os.replace(tmp, target)
    return target


def _run_drumsep(files, output_dir, output_format, params, template):
    import torch
    from demucs.apply import apply_model
    from demucs.pretrained import get_model

    release_model()
    repo = MODELS_DIR / "drumsep"
    repo.mkdir(parents=True, exist_ok=True)
    try:
        _download_file(DRUMSEP_URL, repo / "drumsep.th")
        model = get_model("drumsep", repo=repo)
    except Exception as e:
        raise UserError(f"Could not load drumsep: {describe_error(e)}") from None
    device = "cuda" if cuda_available() else "cpu"
    norm = float(params.get("normalization_threshold", 0.9))
    amp = float(params.get("amplification_threshold", 0.7))
    print(f"Model loaded: drumsep ({device}). Stems: {', '.join(model.sources)}")

    start, failed = time.time(), []
    for i, file in enumerate(files, 1):
        print(f"\n[{i}/{len(files)}] Separating: {file.name}")
        try:
            wave = load_audio(file, model.samplerate)
            if wave.shape[0] == 1:
                wave = wave.repeat(model.audio_channels, axis=0)
            mix = torch.from_numpy(wave[:model.audio_channels].copy())
            ref = mix.mean(0)
            mean, std = ref.mean(), ref.std() + 1e-8
            with torch.no_grad():
                sources = apply_model(model, ((mix - mean) / std)[None], device=device, shifts=int(params["shifts"]),
                                      split=True, overlap=float(params["overlap"]), progress=True)[0]
            sources = (sources * std + mean).cpu().numpy()
            for stem, source in zip(model.sources, sources):
                name = render_name(template, input=file.stem, stem=stem.title(), model="drumsep")
                out = write_audio(output_dir / f"{name}.{output_format}", normalize_peak(source, norm, amp),
                                  model.samplerate)
                print(f"   ✔ {out.name}")
        except Exception as e:
            failed.append(file.name)
            print(f"   ❌ Error separating {file.name}: {describe_error(e)}")
    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"\nDone: {len(files) - len(failed)} separated, {len(failed)} failed in "
          f"{format_time(time.time() - start)}. Output: {output_dir}")


# ---------- ensemble ----------
ENSEMBLE_ALGORITHMS = ("avg_wave", "median_wave", "min_wave", "max_wave", "avg_fft", "median_fft", "min_fft",
                       "max_fft", "uvr_max_spec", "uvr_min_spec", "ensemble_wav")
WEIGHTED_ALGORITHMS = {"avg_wave", "avg_fft"}
ENSEMBLE_SAMPLE_RATE = 44100
ENSEMBLE_GROUP_SIZE = 8  # files separated by every model before their stems are combined
STEM_NAME_MAP = {
    "vocals": "Vocals", "instrumental": "Instrumental", "inst": "Instrumental", "karaoke": "Instrumental",
    "other": "Other", "no_vocals": "Instrumental", "no vocals": "Instrumental", "drums": "Drums", "bass": "Bass",
    "guitar": "Guitar", "piano": "Piano", "synthesizer": "Synthesizer", "strings": "Strings",
    "woodwinds": "Woodwinds", "brass": "Brass", "wind inst": "Wind Inst", "lead vocals": "Lead Vocals",
    "backing vocals": "Backing Vocals", "primary stem": "Primary Stem", "secondary stem": "Secondary Stem",
}


def model_label(arch, model_id):
    return f"[{ARCH_LABELS[arch]}] {model_id}"


def find_model_by_label(label):
    for arch, models in MODELS.items():
        for model_id in models:
            if model_label(arch, model_id) == label:
                return arch, model_id
    raise UserError(f"Unknown model: {label}")


def find_model_by_filename(filename):
    for arch, models in MODELS.items():
        for model_id, model_file in models.items():
            if model_file == filename:
                return arch, model_id
    raise UserError(f"The model {filename} is not in the model list")


def load_ensemble_presets():
    from importlib import resources

    text = resources.files("audio_separator").joinpath("ensemble_presets.json").read_text(encoding="utf-8")
    return json.loads(text)["presets"]


def canonical_stem(raw, model_stems):
    """Name a model's stem so it matches the same stem of other models ("vocals" == "Vocals")."""
    lower = raw.strip().lower()
    if "vocal" in lower and not lower.startswith("no") and "lead" not in lower and "backing" not in lower:
        return "Vocals"
    if lower == "other" and len(model_stems) == 2 and any("vocal" in s.lower() for s in model_stems):
        return "Instrumental"  # a two-stem vocal model's "other" is the instrumental
    return STEM_NAME_MAP.get(lower) or raw.strip().title()


def parse_weights(text, count):
    text = (text or "").strip()
    if not text:
        return None
    try:
        weights = [float(w) for w in re.split(r"[,\s]+", text) if w]
    except ValueError:
        raise UserError("Weights must be numbers separated by commas, e.g. 1, 2") from None
    if len(weights) != count:
        raise UserError(f"There must be one weight per model ({count} models, {len(weights)} weights)")
    if not all(math.isfinite(w) and w >= 0 for w in weights) or sum(weights) <= 0:
        raise UserError("Weights must be non-negative numbers that add up to more than zero")
    return weights


def run_ensemble(preset, models, algorithm, weights, audio_input, output_folder, output_format,
                 include_unmatched=False, keep_intermediates=False, normalization_threshold=0.9,
                 amplification_threshold=0.7):
    """Separate with 2-6 models and combine their stems with an ensemble algorithm."""
    try:
        _run_ensemble(preset, models, algorithm, weights, audio_input, output_folder, output_format,
                      include_unmatched, keep_intermediates, normalization_threshold, amplification_threshold)
    except UserError as e:
        print_error(e)


def _run_ensemble(preset, models, algorithm, weights, audio_input, output_folder, output_format,
                  include_unmatched, keep_intermediates, norm, amp):
    if preset and preset != "custom":
        presets = load_ensemble_presets()
        if preset not in presets:
            raise UserError(f"Unknown preset {preset}. Available: {', '.join(presets)}")
        info = presets[preset]
        members = [find_model_by_filename(f) for f in info["models"]]
        algorithm, weight_values, label = info["algorithm"], info.get("weights"), preset
        print(f"Preset: {info.get('name', preset)} ({algorithm})")
        if info.get("description"):
            print(f"   {info['description']}")
    else:
        members = [find_model_by_label(m) for m in models if m and m != "None"]
        if algorithm not in ENSEMBLE_ALGORITHMS:
            raise UserError(f"Unknown ensemble algorithm: {algorithm}")
        weight_values = parse_weights(weights, len(members)) if algorithm in WEIGHTED_ALGORITHMS else None
        label = algorithm
    if not 2 <= len(members) <= 6:
        raise UserError("Select between 2 and 6 models for the ensemble")
    if len(set(members)) != len(members):
        raise UserError("The same model is selected more than once")
    template = SETTINGS["name_template"]
    check_name_template(template)

    output_dir = resolve_output(output_folder)
    files, temp = resolve_inputs(audio_input)
    try:
        if not files:
            print("No valid audio files found.")
            return
        print(f"{len(files)} audio file(s) found. Models: " + ", ".join(model_id for _, model_id in members))
        start, failed = time.time(), []
        for g in range(0, len(files), ENSEMBLE_GROUP_SIZE):
            group = files[g:g + ENSEMBLE_GROUP_SIZE]
            failed += _ensemble_group(group, members, algorithm, weight_values, label, output_dir, output_format,
                                      include_unmatched, keep_intermediates, norm, amp, template)
        print(f"\nDone: {len(files) - len(failed)} ensembled, {len(failed)} failed in "
              f"{format_time(time.time() - start)}. Output: {output_dir}")
    finally:
        if temp:
            shutil.rmtree(temp, ignore_errors=True)


def _ensemble_group(group, members, algorithm, weights, label, output_dir, output_format, include_unmatched,
                    keep_intermediates, norm, amp, template):
    work_dir = TEMP_DIR / "ensemble"
    shutil.rmtree(work_dir, ignore_errors=True)
    outputs = {file: [] for file in group}  # per file, the outputs of each member
    failed = {}
    try:
        # Model by model, so every model is loaded once per group of files
        for index, (arch, model_id) in enumerate(members, 1):
            member_dir = work_dir / f"model_{index}"
            member_dir.mkdir(parents=True, exist_ok=True)
            params = {**ARCH_DEFAULTS[arch], **LAST_PARAMS.get(arch, {}), "single_stem": ""}
            kwargs = {"output_dir": str(member_dir), "output_format": "wav", **runtime_kwargs(),
                      **build_arch_kwargs(arch, params)}
            print(f"\nLoading model {index}/{len(members)}: {model_id}")
            separator, _ = get_separator(MODELS[arch][model_id], kwargs)
            for file in group:
                if file in failed:
                    continue
                print(f"   Separating {file.name}")
                try:
                    names = separator.separate(str(file))
                    if not names:
                        raise RuntimeError("no output produced")
                    outputs[file].append([member_dir / Path(n).name for n in names])
                except Exception as e:
                    failed[file] = e
                    print(f"   ❌ Error separating {file.name}: {describe_error(e)}")

        for file in group:
            if file in failed:
                continue
            print(f"\nCombining stems of {file.name} ({algorithm})")
            try:
                for out in _combine(file, outputs[file], algorithm, weights, label, output_dir, output_format,
                                    include_unmatched, norm, amp, template):
                    print(f"   ✔ {out.name}")
            except Exception as e:
                failed[file] = e
                print(f"   ❌ Error combining {file.name}: {describe_error(e)}")
            if keep_intermediates:
                target = output_dir / f"{file.stem}_ensemble_parts"
                target.mkdir(parents=True, exist_ok=True)
                for stem_paths in outputs[file]:
                    for p in stem_paths:
                        if p.exists():
                            shutil.move(str(p), str(target / p.name))
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)
    return [f.name for f in failed]


def _combine(file, member_outputs, algorithm, weights, label, output_dir, output_format, include_unmatched,
             norm, amp, template):
    import numpy as np
    from audio_separator.separator.ensembler import Ensembler

    groups = {}
    for index, stem_paths in enumerate(member_outputs):
        raw_names = [parse_stem_label(p.name) or "Unknown" for p in stem_paths]
        for p, raw in zip(stem_paths, raw_names):
            groups.setdefault(canonical_stem(raw, raw_names), []).append((index, p))

    def shared(items):
        return len({index for index, _ in items}) == len(member_outputs)

    if not any(shared(items) for items in groups.values()) and not include_unmatched:
        raise UserError(f"The selected models have no stem in common (stems found: {', '.join(sorted(groups))}). "
                        "Choose models that separate the same stems or enable include_unmatched.")
    results = []
    for stem, items in sorted(groups.items(), key=lambda group: not shared(group[1])):
        if not shared(items) and not include_unmatched:
            continue
        waves = [load_audio(p, ENSEMBLE_SAMPLE_RATE) for _, p in items]
        mono = all(w.shape[0] == 1 for w in waves)
        waves = [np.vstack([w, w]) if w.shape[0] == 1 else w for w in waves]
        stem_weights = [float(weights[index]) for index, _ in items] if weights else None
        combined = Ensembler(logging.getLogger("ensemble"), algorithm, stem_weights).ensemble(waves)
        length = max(w.shape[1] for w in waves)  # spectrogram algorithms can return a few samples less
        combined = np.pad(combined, ((0, 0), (0, max(length - combined.shape[1], 0))))[:, :length]
        if mono:
            combined = combined[:1]
        combined = normalize_peak(combined, norm, amp)
        safe_label = re.sub(r"[^\w.-]+", "_", label).strip("_") or algorithm
        name = render_name(template, input=file.stem, stem=stem, model=f"ensemble_{safe_label}")
        results.append(write_audio(output_dir / f"{name}.{output_format}", combined, ENSEMBLE_SAMPLE_RATE,
                                   subtype=audio_subtype(items[0][1])))
    return results


# ---------- settings and downloader cells ----------
def apply_settings(precision, torch_compile, chunk_duration, use_soundfile, name_template):
    try:
        if precision not in PRECISIONS:
            raise UserError(f"Unknown precision: {precision}")
        name_template = name_template.strip() or DEFAULT_NAME_TEMPLATE
        check_name_template(name_template)
        SETTINGS.update(precision=precision, torch_compile=bool(torch_compile),
                        chunk_duration=max(float(chunk_duration or 0), 0.0), use_soundfile=bool(use_soundfile),
                        name_template=name_template)
        release_model()
        print("Settings saved:")
        for key, value in SETTINGS.items():
            print(f"   {key}: {value}")
    except UserError as e:
        print_error(e)


def run_download(url, save_folder, audio_format="wav", audio_name=""):
    try:
        if not is_url(url):
            raise UserError("Paste a valid link (starting with http:// or https://)")
        folder = clean_path(save_folder) or str(Path(path) / "inputs")
        if folder.startswith("/content/drive") and not drive_mounted():
            raise UserError("Google Drive is not mounted: run the Installation cell with mount_drive enabled.")
        name = _UNSAFE_CHARS_RE.sub("_", audio_name.strip()) if audio_name.strip() else None
        result = download_audio(url, folder, audio_format, name)
        print(f"\nDownload complete: {result}")
    except UserError as e:
        print_error(e)


clear_output()
print(f"Environment: {ENV.title()} | GPU: {'yes' if cuda_available() else 'no'}")
print("Installation done!")

# <small> **Settings (optional)**

In [ ]:
#@markdown # **Settings (optional)**
#@markdown Run this cell only to change these options, they apply to every separation cell. Otherwise the defaults are used.

#@markdown Precision: `auto` = autocast on GPU and fp32 on CPU. `native_fp16` is faster on NVIDIA GPUs with BS/Mel Roformer models (may slightly change results).
precision = "auto" #@param ["auto", "fp32", "autocast", "native_fp16"]
#@markdown Compile BS/Mel Roformer models with torch.compile: slower first run, faster afterwards.
torch_compile = False #@param {type:"boolean"}
#@markdown Split long audio into chunks of this many seconds to save memory (0 = off). 600 is recommended for audio over an hour.
chunk_duration = 0 #@param {type:"slider", min:0, max:3600, step:60}
#@markdown Write outputs with soundfile instead of pydub (uses less memory with long files).
use_soundfile = False #@param {type:"boolean"}
#@markdown Output file names. Placeholders: `{input}` input file name, `{stem}` stem name (required), `{model}` model name.
name_template = "{input}_({stem})_{model}" #@param {type:"string"}

apply_settings(precision, torch_compile, chunk_duration, use_soundfile, name_template)

# <small> **Select Separation method:**

### ***Roformer (BS-Roformer and MelBand Roformer)***

In [ ]:
#@markdown # **Separation! Roformer Only**
#@markdown Input: an audio file, a folder (batch: every audio file inside is separated, loading the model only once) or a link:
audio_input = "/content/drive/MyDrive/UVR5/inputs" #@param {type:"string"}
#@markdown You can paste the link to the video/audio from many sites, check the complete list [here](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown Output folder for the separated stems:
output_folder = "/content/drive/MyDrive/UVR5/outputs" #@param {type:"string"}

#@markdown **Recommended:** Instrumental: `BS Roformer | Instrumental Resurrection by unwa` · Vocals: `BS Roformer SW by jarredou` · Karaoke: `BS Roformer | Karaoke by anvuew` · De-Reverb: `MelBand Roformer | De-Reverb by anvuew` · Denoise: `MelBand Roformer | Denoise-Debleed by Gabox`
#@markdown Select the model:
model = "BS Roformer | Instrumental Resurrection by unwa" #@param ["BS-Roformer-Viperx-1297", "BS-Roformer-Viperx-1296", "BS-Roformer-Viperx-1053", "Mel-Roformer-Viperx-1143", "BS-Roformer-De-Reverb", "Mel-Roformer-Crowd-Aufr33-Viperx", "Mel-Roformer-Denoise-Aufr33", "Mel-Roformer-Denoise-Aufr33-Aggr", "MelBand Roformer | Denoise-Debleed by Gabox", "Mel-Roformer-Karaoke-Aufr33-Viperx", "MelBand Roformer | Karaoke by Gabox", "MelBand Roformer | Karaoke V2 by Gabox", "MelBand Roformer | Karaoke by becruily", "BS Roformer | Karaoke by frazer-becruily", "BS Roformer | Karaoke by anvuew", "MelBand Roformer | Vocals by Kimberley Jensen", "MelBand Roformer Kim | FT by unwa", "MelBand Roformer Kim | FT 2 by unwa", "MelBand Roformer Kim | FT 2 Bleedless by unwa", "MelBand Roformer Kim | FT 3 by unwa", "MelBand Roformer Kim | Inst V1 by Unwa", "MelBand Roformer Kim | Inst V1 Plus by Unwa", "MelBand Roformer Kim | Inst V1 (E) by Unwa", "MelBand Roformer Kim | Inst V1 (E) Plus by Unwa", "MelBand Roformer Kim | Inst V2 by Unwa", "MelBand Roformer Kim | InstVoc Duality V1 by Unwa", "MelBand Roformer Kim | InstVoc Duality V2 by Unwa", "MelBand Roformer | Vocals by becruily", "MelBand Roformer | Instrumental by becruily", "MelBand Roformer | Vocals Fullness by Aname", "BS Roformer | Vocals by Gabox", "MelBand Roformer | Vocals by Gabox", "MelBand Roformer | Vocals V2 by Gabox", "MelBand Roformer | Vocals FV1 by Gabox", "MelBand Roformer | Vocals FV2 by Gabox", "MelBand Roformer | Vocals FV3 by Gabox", "MelBand Roformer | Vocals FV4 by Gabox", "MelBand Roformer | Vocals FV5 by Gabox", "MelBand Roformer | Vocals FV6 by Gabox", "MelBand Roformer | Vocals FV7b by Gabox", "MelBand Roformer | Instrumental by Gabox", "MelBand Roformer | Instrumental 2 by Gabox", "MelBand Roformer | Instrumental 3 by Gabox", "MelBand Roformer | Instrumental Bleedless V1 by Gabox", "MelBand Roformer | Instrumental Bleedless V2 by Gabox", "MelBand Roformer | Instrumental Bleedless V3 by Gabox", "MelBand Roformer | Instrumental Fullness V1 by Gabox", "MelBand Roformer | Instrumental Fullness V2 by Gabox", "MelBand Roformer | Instrumental Fullness V3 by Gabox", "MelBand Roformer | Instrumental Fullness V4 by Gabox", "MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox", "MelBand Roformer | INSTV5 by Gabox", "MelBand Roformer | INSTV5N by Gabox", "MelBand Roformer | INSTV6 by Gabox", "MelBand Roformer | INSTV6N by Gabox", "MelBand Roformer | INSTV7 by Gabox", "MelBand Roformer | INSTV7N by Gabox", "MelBand Roformer | INSTV8 by Gabox", "MelBand Roformer | INSTV8N by Gabox", "MelBand Roformer | Instrumental FV7z by Gabox", "MelBand Roformer | Instrumental FV8 by Gabox", "MelBand Roformer | Instrumental FV8b by Gabox", "MelBand Roformer | Instrumental FVX by Gabox", "MelBand Roformer | De-Reverb by anvuew", "MelBand Roformer | De-Reverb Less Aggressive by anvuew", "MelBand Roformer | De-Reverb Mono by anvuew", "MelBand Roformer | De-Reverb Big by Sucial", "MelBand Roformer | De-Reverb Super Big by Sucial", "MelBand Roformer | De-Reverb-Echo by Sucial", "MelBand Roformer | De-Reverb-Echo V2 by Sucial", "MelBand Roformer | De-Reverb-Echo Fused by Sucial", "MelBand Roformer Kim | SYHFT by SYH99999", "MelBand Roformer Kim | SYHFT V2 by SYH99999", "MelBand Roformer Kim | SYHFT V2.5 by SYH99999", "MelBand Roformer Kim | SYHFT V3 by SYH99999", "MelBand Roformer Kim | Big SYHFT V1 by SYH99999", "MelBand Roformer Kim | Big Beta 4 FT by unwa", "MelBand Roformer Kim | Big Beta 5e FT by unwa", "MelBand Roformer | Big Beta 6 by unwa", "MelBand Roformer | Big Beta 6X by unwa", "BS Roformer | Vocals Revive by Unwa", "BS Roformer | Vocals Revive V2 by Unwa", "BS Roformer | Vocals Revive V3e by Unwa", "BS Roformer | Chorus Male-Female by Sucial", "BS Roformer | Male-Female by aufr33", "MelBand Roformer | Aspiration by Sucial", "MelBand Roformer | Aspiration Less Aggressive by Sucial", "MelBand Roformer | Bleed Suppressor V1 by unwa-97chris", "BS Roformer | Vocals Resurrection by unwa", "BS Roformer | Instrumental Resurrection by unwa", "BS Roformer | Instrumental ResurrectioN by Gabox", "BS Roformer SW by jarredou"]
#@markdown Select the output format:
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]

#@markdown ### **Model settings**
#@markdown Larger consumes more resources, but may give better results.
segment_size = 256 #@param {type:"slider", min:32, max:4000, step:32}
#@markdown Override the model default segment size instead of using the model default value.
override_model_segment_size = False #@param {type:"boolean"}
#@markdown Amount of overlap between prediction windows.
overlap = 8 #@param {type:"slider", min:2, max:10, step:1}
#@markdown Ignore the overlap above and use the overlap of the model config.
use_model_overlap = False #@param {type:"boolean"}
#@markdown Shift the pitch of the input before separating (semitones), then shift the stems back. Can help with very high or low voices.
pitch_shift = 0 #@param {type:"slider", min:-12, max:12, step:1}
#@markdown ### **Output levels**
#@markdown Max peak of the output: louder outputs are scaled down to it.
normalization_threshold = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Min peak of the output: quieter outputs are scaled up to it.
amplification_threshold = 0.7 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Output only one stem (e.g. Instrumental, Vocals, Drums...). Leave empty to output all stems.
single_stem = "" #@param {type:"string"}

ARCH = "roformer"
run_separation(ARCH, model, audio_input, output_folder, output_format, dict(
    segment_size=segment_size, override_model_segment_size=override_model_segment_size, overlap=overlap,
    use_model_overlap=use_model_overlap, batch_size=1, pitch_shift=pitch_shift,
    normalization_threshold=normalization_threshold, amplification_threshold=amplification_threshold,
    single_stem=single_stem))

### ***MDX23C***

In [ ]:
#@markdown # **Separation! MDX23C Only**
#@markdown Input: an audio file, a folder (batch: every audio file inside is separated, loading the model only once) or a link:
audio_input = "/content/drive/MyDrive/UVR5/inputs" #@param {type:"string"}
#@markdown You can paste the link to the video/audio from many sites, check the complete list [here](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown Output folder for the separated stems:
output_folder = "/content/drive/MyDrive/UVR5/outputs" #@param {type:"string"}

#@markdown Select the model:
model = "MDX23C-8KFFT-InstVoc_HQ_2.ckpt" #@param ["MDX23C_D1581.ckpt", "MDX23C-8KFFT-InstVoc_HQ.ckpt", "MDX23C-8KFFT-InstVoc_HQ_2.ckpt", "MDX23C-De-Reverb-aufr33-jarredou.ckpt", "MDX23C-DrumSep-aufr33-jarredou.ckpt"]
#@markdown Select the output format:
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]

#@markdown ### **Model settings**
#@markdown Larger consumes more resources, but may give better results.
segment_size = 256 #@param {type:"slider", min:32, max:4000, step:32}
#@markdown Override the model default segment size instead of using the model default value.
override_model_segment_size = False #@param {type:"boolean"}
#@markdown Amount of overlap between prediction windows.
overlap = 8 #@param {type:"slider", min:2, max:50, step:1}
#@markdown Ignore the overlap above and use the overlap of the model config.
use_model_overlap = False #@param {type:"boolean"}
#@markdown Shift the pitch of the input before separating (semitones), then shift the stems back. Can help with very high or low voices.
pitch_shift = 0 #@param {type:"slider", min:-12, max:12, step:1}
#@markdown Larger consumes more RAM but may process slightly faster.
batch_size = 1 #@param {type:"slider", min:1, max:16, step:1}
#@markdown ### **Output levels**
#@markdown Max peak of the output: louder outputs are scaled down to it.
normalization_threshold = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Min peak of the output: quieter outputs are scaled up to it.
amplification_threshold = 0.7 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Output only one stem (e.g. Instrumental, Vocals, Drums...). Leave empty to output all stems.
single_stem = "" #@param {type:"string"}

ARCH = "mdx23c"
run_separation(ARCH, model, audio_input, output_folder, output_format, dict(
    segment_size=segment_size, override_model_segment_size=override_model_segment_size, overlap=overlap,
    use_model_overlap=use_model_overlap, batch_size=batch_size, pitch_shift=pitch_shift,
    normalization_threshold=normalization_threshold, amplification_threshold=amplification_threshold,
    single_stem=single_stem))

### ***MDX-NET***

In [ ]:
#@markdown # **Separation! MDX-NET**
#@markdown Input: an audio file, a folder (batch: every audio file inside is separated, loading the model only once) or a link:
audio_input = "/content/drive/MyDrive/UVR5/inputs" #@param {type:"string"}
#@markdown You can paste the link to the video/audio from many sites, check the complete list [here](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown Output folder for the separated stems:
output_folder = "/content/drive/MyDrive/UVR5/outputs" #@param {type:"string"}

#@markdown Select the model:
model = "UVR-MDX-NET-Inst_HQ_5.onnx" #@param ["UVR-MDX-NET-Inst_full_292.onnx", "UVR-MDX-NET_Inst_187_beta.onnx", "UVR-MDX-NET_Inst_82_beta.onnx", "UVR-MDX-NET_Inst_90_beta.onnx", "UVR-MDX-NET_Main_340.onnx", "UVR-MDX-NET_Main_390.onnx", "UVR-MDX-NET_Main_406.onnx", "UVR-MDX-NET_Main_427.onnx", "UVR-MDX-NET_Main_438.onnx", "UVR-MDX-NET-Inst_HQ_1.onnx", "UVR-MDX-NET-Inst_HQ_2.onnx", "UVR-MDX-NET-Inst_HQ_3.onnx", "UVR-MDX-NET-Inst_HQ_4.onnx", "UVR-MDX-NET-Inst_HQ_5.onnx", "UVR_MDXNET_Main.onnx", "UVR-MDX-NET-Inst_Main.onnx", "UVR_MDXNET_1_9703.onnx", "UVR_MDXNET_2_9682.onnx", "UVR_MDXNET_3_9662.onnx", "UVR-MDX-NET-Inst_1.onnx", "UVR-MDX-NET-Inst_2.onnx", "UVR-MDX-NET-Inst_3.onnx", "UVR_MDXNET_KARA.onnx", "UVR_MDXNET_KARA_2.onnx", "UVR_MDXNET_9482.onnx", "UVR-MDX-NET-Voc_FT.onnx", "Kim_Vocal_1.onnx", "Kim_Vocal_2.onnx", "Kim_Inst.onnx", "Reverb_HQ_By_FoxJoy.onnx", "UVR-MDX-NET_Crowd_HQ_1.onnx", "kuielab_a_vocals.onnx", "kuielab_a_other.onnx", "kuielab_a_bass.onnx", "kuielab_a_drums.onnx", "kuielab_b_vocals.onnx", "kuielab_b_other.onnx", "kuielab_b_bass.onnx", "kuielab_b_drums.onnx"]
#@markdown Select the output format:
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]

#@markdown ### **Model settings**
#@markdown Usually called stride in neural networks; only change if you know what you're doing.
hop_length = 1024 #@param {type:"slider", min:32, max:2048, step:32}
#@markdown Larger consumes more resources, but may give better results.
segment_size = 256 #@param {type:"slider", min:32, max:4000, step:32}
#@markdown Amount of overlap between prediction windows.
overlap = 0.25 #@param {type:"slider", min:0.001, max:0.999, step:0.001}
#@markdown Larger consumes more RAM but may process slightly faster.
batch_size = 1 #@param {type:"slider", min:1, max:16, step:1}
#@markdown Enable denoising during separation.
denoise = True #@param {type:"boolean"}
#@markdown Invert the secondary stem using the spectrogram instead of the waveform. Slightly slower, may be more accurate.
invert_using_spec = False #@param {type:"boolean"}
#@markdown ### **Output levels**
#@markdown Max peak of the output: louder outputs are scaled down to it.
normalization_threshold = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Min peak of the output: quieter outputs are scaled up to it.
amplification_threshold = 0.7 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Output only one stem (e.g. Instrumental, Vocals, Drums...). Leave empty to output all stems.
single_stem = "" #@param {type:"string"}

ARCH = "mdxnet"
run_separation(ARCH, model, audio_input, output_folder, output_format, dict(
    hop_length=hop_length, segment_size=segment_size, overlap=overlap, batch_size=batch_size,
    enable_denoise=denoise, invert_using_spec=invert_using_spec, normalization_threshold=normalization_threshold,
    amplification_threshold=amplification_threshold, single_stem=single_stem))

### ***VR ARCH***

In [ ]:
#@markdown # **Separation! VR Arch**
#@markdown Input: an audio file, a folder (batch: every audio file inside is separated, loading the model only once) or a link:
audio_input = "/content/drive/MyDrive/UVR5/inputs" #@param {type:"string"}
#@markdown You can paste the link to the video/audio from many sites, check the complete list [here](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown Output folder for the separated stems:
output_folder = "/content/drive/MyDrive/UVR5/outputs" #@param {type:"string"}

#@markdown Select the model:
model = "UVR-DeEcho-DeReverb.pth" #@param ["1_HP-UVR.pth", "2_HP-UVR.pth", "3_HP-Vocal-UVR.pth", "4_HP-Vocal-UVR.pth", "5_HP-Karaoke-UVR.pth", "6_HP-Karaoke-UVR.pth", "7_HP2-UVR.pth", "8_HP2-UVR.pth", "9_HP2-UVR.pth", "10_SP-UVR-2B-32000-1.pth", "11_SP-UVR-2B-32000-2.pth", "12_SP-UVR-3B-44100.pth", "13_SP-UVR-4B-44100-1.pth", "14_SP-UVR-4B-44100-2.pth", "15_SP-UVR-MID-44100-1.pth", "16_SP-UVR-MID-44100-2.pth", "17_HP-Wind_Inst-UVR.pth", "UVR-De-Echo-Aggressive.pth", "UVR-De-Echo-Normal.pth", "UVR-DeEcho-DeReverb.pth", "UVR-De-Reverb-aufr33-jarredou.pth", "UVR-DeNoise-Lite.pth", "UVR-DeNoise.pth", "UVR-BVE-4B_SN-44100-1.pth", "UVR-BVE-4B_SN-44100-2.pth", "MGM_HIGHEND_v4.pth", "MGM_LOWEND_A_v4.pth", "MGM_LOWEND_B_v4.pth", "MGM_MAIN_v4.pth"]
#@markdown Select the output format:
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]

#@markdown ### **Model settings**
#@markdown Lower window size yields a higher output quality, but will take longer to process.
window_size = 512 #@param {type:"slider", min:320, max:1024, step:32}
#@markdown Intensity of primary stem extraction.
aggression = 5 #@param {type:"slider", min:1, max:50, step:1}
#@markdown Larger consumes more RAM but may process slightly faster.
batch_size = 1 #@param {type:"slider", min:1, max:16, step:1}
#@markdown Enable Test-Time-Augmentation; slow but improves quality.
tta = True #@param {type:"boolean"}
#@markdown Identify leftover artifacts within vocal output; may improve separation for some songs.
post_process = False #@param {type:"boolean"}
#@markdown Threshold for the post process feature.
post_process_threshold = 0.2 #@param {type:"slider", min:0.1, max:0.3, step:0.1}
#@markdown Mirror the missing frequency range of the output.
high_end_process = False #@param {type:"boolean"}
#@markdown ### **Output levels**
#@markdown Max peak of the output: louder outputs are scaled down to it.
normalization_threshold = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Min peak of the output: quieter outputs are scaled up to it.
amplification_threshold = 0.7 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Output only one stem (e.g. Instrumental, Vocals, Drums...). Leave empty to output all stems.
single_stem = "" #@param {type:"string"}

ARCH = "vrarch"
run_separation(ARCH, model, audio_input, output_folder, output_format, dict(
    window_size=window_size, aggression=aggression, batch_size=batch_size, enable_tta=tta,
    enable_post_process=post_process, post_process_threshold=post_process_threshold,
    high_end_process=high_end_process, normalization_threshold=normalization_threshold,
    amplification_threshold=amplification_threshold, single_stem=single_stem))

### ***Demucs***

In [ ]:
#@markdown # **Separation! Demucs**
#@markdown Input: an audio file, a folder (batch: every audio file inside is separated, loading the model only once) or a link:
audio_input = "/content/drive/MyDrive/UVR5/inputs" #@param {type:"string"}
#@markdown You can paste the link to the video/audio from many sites, check the complete list [here](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown Output folder for the separated stems:
output_folder = "/content/drive/MyDrive/UVR5/outputs" #@param {type:"string"}

#@markdown Select the model (drumsep separates a drums stem into kick, snare, cymbals and toms; `MDX23C-DrumSep-aufr33-jarredou.ckpt` in the MDX23C cell is a newer alternative):
model = "htdemucs_ft.yaml" #@param ["htdemucs_ft.yaml", "htdemucs_6s.yaml", "htdemucs.yaml", "hdemucs_mmi.yaml", "drumsep"]
#@markdown Select the output format:
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]

#@markdown ### **Model settings**
#@markdown Number of predictions with random shifts, higher = slower but better quality.
shifts = 2 #@param {type:"slider", min:1, max:20, step:1}
#@markdown Larger consumes more resources, but may give better results (not used by drumsep).
segment_size = 40 #@param {type:"slider", min:1, max:100, step:1}
#@markdown Split the audio into segments of the size above (not used by drumsep).
segments_enabled = True #@param {type:"boolean"}
#@markdown Amount of overlap between prediction windows.
overlap = 0.25 #@param {type:"slider", min:0.001, max:0.999, step:0.001}
#@markdown ### **Output levels**
#@markdown Max peak of the output: louder outputs are scaled down to it.
normalization_threshold = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Min peak of the output: quieter outputs are scaled up to it.
amplification_threshold = 0.7 #@param {type:"slider", min:0.1, max:1, step:0.1}

ARCH = "demucs"
run_separation(ARCH, model, audio_input, output_folder, output_format, dict(
    shifts=shifts, segment_size=segment_size, segments_enabled=segments_enabled, overlap=overlap,
    normalization_threshold=normalization_threshold, amplification_threshold=amplification_threshold))

### ***Ensemble***

In [ ]:
#@markdown # **Ensemble!**
#@markdown Separate with several models and combine their stems. Each model uses the settings of the last separation run with its architecture (or the defaults).
#@markdown Input: an audio file, a folder (batch: every audio file inside is separated, loading the model only once) or a link:
audio_input = "/content/drive/MyDrive/UVR5/inputs" #@param {type:"string"}
#@markdown You can paste the link to the video/audio from many sites, check the complete list [here](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown Output folder for the separated stems:
output_folder = "/content/drive/MyDrive/UVR5/outputs" #@param {type:"string"}

#@markdown Select a preset, or "custom" to choose the models and the algorithm below:
preset = "vocal_balanced" #@param ["custom", "instrumental_clean", "instrumental_full", "instrumental_balanced", "instrumental_low_resource", "vocal_balanced", "vocal_clean", "vocal_full", "vocal_rvc", "karaoke"]
#@markdown **Presets** (from audio-separator): `instrumental_clean` cleanest instrumentals · `instrumental_full` maximum instrument preservation · `instrumental_balanced` balance of noise and fullness · `instrumental_low_resource` fast, low VRAM · `vocal_balanced` best overall vocals · `vocal_clean` minimal instrument bleed · `vocal_full` maximum vocal capture · `vocal_rvc` for RVC/AI training · `karaoke` lead vocal removal.

#@markdown ### **Custom ensemble** (only used when preset is "custom", select 2 to 4 models)
model_1 = "[Roformer] BS Roformer SW by jarredou" #@param ["None", "[Roformer] BS-Roformer-Viperx-1297", "[Roformer] BS-Roformer-Viperx-1296", "[Roformer] BS-Roformer-Viperx-1053", "[Roformer] Mel-Roformer-Viperx-1143", "[Roformer] BS-Roformer-De-Reverb", "[Roformer] Mel-Roformer-Crowd-Aufr33-Viperx", "[Roformer] Mel-Roformer-Denoise-Aufr33", "[Roformer] Mel-Roformer-Denoise-Aufr33-Aggr", "[Roformer] MelBand Roformer | Denoise-Debleed by Gabox", "[Roformer] Mel-Roformer-Karaoke-Aufr33-Viperx", "[Roformer] MelBand Roformer | Karaoke by Gabox", "[Roformer] MelBand Roformer | Karaoke V2 by Gabox", "[Roformer] MelBand Roformer | Karaoke by becruily", "[Roformer] BS Roformer | Karaoke by frazer-becruily", "[Roformer] BS Roformer | Karaoke by anvuew", "[Roformer] MelBand Roformer | Vocals by Kimberley Jensen", "[Roformer] MelBand Roformer Kim | FT by unwa", "[Roformer] MelBand Roformer Kim | FT 2 by unwa", "[Roformer] MelBand Roformer Kim | FT 2 Bleedless by unwa", "[Roformer] MelBand Roformer Kim | FT 3 by unwa", "[Roformer] MelBand Roformer Kim | Inst V1 by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V2 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V1 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V2 by Unwa", "[Roformer] MelBand Roformer | Vocals by becruily", "[Roformer] MelBand Roformer | Instrumental by becruily", "[Roformer] MelBand Roformer | Vocals Fullness by Aname", "[Roformer] BS Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals V2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV1 by Gabox", "[Roformer] MelBand Roformer | Vocals FV2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV3 by Gabox", "[Roformer] MelBand Roformer | Vocals FV4 by Gabox", "[Roformer] MelBand Roformer | Vocals FV5 by Gabox", "[Roformer] MelBand Roformer | Vocals FV6 by Gabox", "[Roformer] MelBand Roformer | Vocals FV7b by Gabox", "[Roformer] MelBand Roformer | Instrumental by Gabox", "[Roformer] MelBand Roformer | Instrumental 2 by Gabox", "[Roformer] MelBand Roformer | Instrumental 3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V4 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox", "[Roformer] MelBand Roformer | INSTV5 by Gabox", "[Roformer] MelBand Roformer | INSTV5N by Gabox", "[Roformer] MelBand Roformer | INSTV6 by Gabox", "[Roformer] MelBand Roformer | INSTV6N by Gabox", "[Roformer] MelBand Roformer | INSTV7 by Gabox", "[Roformer] MelBand Roformer | INSTV7N by Gabox", "[Roformer] MelBand Roformer | INSTV8 by Gabox", "[Roformer] MelBand Roformer | INSTV8N by Gabox", "[Roformer] MelBand Roformer | Instrumental FV7z by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8 by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8b by Gabox", "[Roformer] MelBand Roformer | Instrumental FVX by Gabox", "[Roformer] MelBand Roformer | De-Reverb by anvuew", "[Roformer] MelBand Roformer | De-Reverb Less Aggressive by anvuew", "[Roformer] MelBand Roformer | De-Reverb Mono by anvuew", "[Roformer] MelBand Roformer | De-Reverb Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb Super Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo V2 by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo Fused by Sucial", "[Roformer] MelBand Roformer Kim | SYHFT by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2.5 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V3 by SYH99999", "[Roformer] MelBand Roformer Kim | Big SYHFT V1 by SYH99999", "[Roformer] MelBand Roformer Kim | Big Beta 4 FT by unwa", "[Roformer] MelBand Roformer Kim | Big Beta 5e FT by unwa", "[Roformer] MelBand Roformer | Big Beta 6 by unwa", "[Roformer] MelBand Roformer | Big Beta 6X by unwa", "[Roformer] BS Roformer | Vocals Revive by Unwa", "[Roformer] BS Roformer | Vocals Revive V2 by Unwa", "[Roformer] BS Roformer | Vocals Revive V3e by Unwa", "[Roformer] BS Roformer | Chorus Male-Female by Sucial", "[Roformer] BS Roformer | Male-Female by aufr33", "[Roformer] MelBand Roformer | Aspiration by Sucial", "[Roformer] MelBand Roformer | Aspiration Less Aggressive by Sucial", "[Roformer] MelBand Roformer | Bleed Suppressor V1 by unwa-97chris", "[Roformer] BS Roformer | Vocals Resurrection by unwa", "[Roformer] BS Roformer | Instrumental Resurrection by unwa", "[Roformer] BS Roformer | Instrumental ResurrectioN by Gabox", "[Roformer] BS Roformer SW by jarredou", "[MDX23C] MDX23C_D1581.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ_2.ckpt", "[MDX23C] MDX23C-De-Reverb-aufr33-jarredou.ckpt", "[MDX23C] MDX23C-DrumSep-aufr33-jarredou.ckpt", "[MDX-NET] UVR-MDX-NET-Inst_full_292.onnx", "[MDX-NET] UVR-MDX-NET_Inst_187_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_82_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_90_beta.onnx", "[MDX-NET] UVR-MDX-NET_Main_340.onnx", "[MDX-NET] UVR-MDX-NET_Main_390.onnx", "[MDX-NET] UVR-MDX-NET_Main_406.onnx", "[MDX-NET] UVR-MDX-NET_Main_427.onnx", "[MDX-NET] UVR-MDX-NET_Main_438.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_3.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_4.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_5.onnx", "[MDX-NET] UVR_MDXNET_Main.onnx", "[MDX-NET] UVR-MDX-NET-Inst_Main.onnx", "[MDX-NET] UVR_MDXNET_1_9703.onnx", "[MDX-NET] UVR_MDXNET_2_9682.onnx", "[MDX-NET] UVR_MDXNET_3_9662.onnx", "[MDX-NET] UVR-MDX-NET-Inst_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_3.onnx", "[MDX-NET] UVR_MDXNET_KARA.onnx", "[MDX-NET] UVR_MDXNET_KARA_2.onnx", "[MDX-NET] UVR_MDXNET_9482.onnx", "[MDX-NET] UVR-MDX-NET-Voc_FT.onnx", "[MDX-NET] Kim_Vocal_1.onnx", "[MDX-NET] Kim_Vocal_2.onnx", "[MDX-NET] Kim_Inst.onnx", "[MDX-NET] Reverb_HQ_By_FoxJoy.onnx", "[MDX-NET] UVR-MDX-NET_Crowd_HQ_1.onnx", "[MDX-NET] kuielab_a_vocals.onnx", "[MDX-NET] kuielab_a_other.onnx", "[MDX-NET] kuielab_a_bass.onnx", "[MDX-NET] kuielab_a_drums.onnx", "[MDX-NET] kuielab_b_vocals.onnx", "[MDX-NET] kuielab_b_other.onnx", "[MDX-NET] kuielab_b_bass.onnx", "[MDX-NET] kuielab_b_drums.onnx", "[VR Arch] 1_HP-UVR.pth", "[VR Arch] 2_HP-UVR.pth", "[VR Arch] 3_HP-Vocal-UVR.pth", "[VR Arch] 4_HP-Vocal-UVR.pth", "[VR Arch] 5_HP-Karaoke-UVR.pth", "[VR Arch] 6_HP-Karaoke-UVR.pth", "[VR Arch] 7_HP2-UVR.pth", "[VR Arch] 8_HP2-UVR.pth", "[VR Arch] 9_HP2-UVR.pth", "[VR Arch] 10_SP-UVR-2B-32000-1.pth", "[VR Arch] 11_SP-UVR-2B-32000-2.pth", "[VR Arch] 12_SP-UVR-3B-44100.pth", "[VR Arch] 13_SP-UVR-4B-44100-1.pth", "[VR Arch] 14_SP-UVR-4B-44100-2.pth", "[VR Arch] 15_SP-UVR-MID-44100-1.pth", "[VR Arch] 16_SP-UVR-MID-44100-2.pth", "[VR Arch] 17_HP-Wind_Inst-UVR.pth", "[VR Arch] UVR-De-Echo-Aggressive.pth", "[VR Arch] UVR-De-Echo-Normal.pth", "[VR Arch] UVR-DeEcho-DeReverb.pth", "[VR Arch] UVR-De-Reverb-aufr33-jarredou.pth", "[VR Arch] UVR-DeNoise-Lite.pth", "[VR Arch] UVR-DeNoise.pth", "[VR Arch] UVR-BVE-4B_SN-44100-1.pth", "[VR Arch] UVR-BVE-4B_SN-44100-2.pth", "[VR Arch] MGM_HIGHEND_v4.pth", "[VR Arch] MGM_LOWEND_A_v4.pth", "[VR Arch] MGM_LOWEND_B_v4.pth", "[VR Arch] MGM_MAIN_v4.pth", "[Demucs] htdemucs_ft.yaml", "[Demucs] htdemucs_6s.yaml", "[Demucs] htdemucs.yaml", "[Demucs] hdemucs_mmi.yaml"]
model_2 = "[Roformer] MelBand Roformer | Big Beta 6X by unwa" #@param ["None", "[Roformer] BS-Roformer-Viperx-1297", "[Roformer] BS-Roformer-Viperx-1296", "[Roformer] BS-Roformer-Viperx-1053", "[Roformer] Mel-Roformer-Viperx-1143", "[Roformer] BS-Roformer-De-Reverb", "[Roformer] Mel-Roformer-Crowd-Aufr33-Viperx", "[Roformer] Mel-Roformer-Denoise-Aufr33", "[Roformer] Mel-Roformer-Denoise-Aufr33-Aggr", "[Roformer] MelBand Roformer | Denoise-Debleed by Gabox", "[Roformer] Mel-Roformer-Karaoke-Aufr33-Viperx", "[Roformer] MelBand Roformer | Karaoke by Gabox", "[Roformer] MelBand Roformer | Karaoke V2 by Gabox", "[Roformer] MelBand Roformer | Karaoke by becruily", "[Roformer] BS Roformer | Karaoke by frazer-becruily", "[Roformer] BS Roformer | Karaoke by anvuew", "[Roformer] MelBand Roformer | Vocals by Kimberley Jensen", "[Roformer] MelBand Roformer Kim | FT by unwa", "[Roformer] MelBand Roformer Kim | FT 2 by unwa", "[Roformer] MelBand Roformer Kim | FT 2 Bleedless by unwa", "[Roformer] MelBand Roformer Kim | FT 3 by unwa", "[Roformer] MelBand Roformer Kim | Inst V1 by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V2 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V1 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V2 by Unwa", "[Roformer] MelBand Roformer | Vocals by becruily", "[Roformer] MelBand Roformer | Instrumental by becruily", "[Roformer] MelBand Roformer | Vocals Fullness by Aname", "[Roformer] BS Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals V2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV1 by Gabox", "[Roformer] MelBand Roformer | Vocals FV2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV3 by Gabox", "[Roformer] MelBand Roformer | Vocals FV4 by Gabox", "[Roformer] MelBand Roformer | Vocals FV5 by Gabox", "[Roformer] MelBand Roformer | Vocals FV6 by Gabox", "[Roformer] MelBand Roformer | Vocals FV7b by Gabox", "[Roformer] MelBand Roformer | Instrumental by Gabox", "[Roformer] MelBand Roformer | Instrumental 2 by Gabox", "[Roformer] MelBand Roformer | Instrumental 3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V4 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox", "[Roformer] MelBand Roformer | INSTV5 by Gabox", "[Roformer] MelBand Roformer | INSTV5N by Gabox", "[Roformer] MelBand Roformer | INSTV6 by Gabox", "[Roformer] MelBand Roformer | INSTV6N by Gabox", "[Roformer] MelBand Roformer | INSTV7 by Gabox", "[Roformer] MelBand Roformer | INSTV7N by Gabox", "[Roformer] MelBand Roformer | INSTV8 by Gabox", "[Roformer] MelBand Roformer | INSTV8N by Gabox", "[Roformer] MelBand Roformer | Instrumental FV7z by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8 by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8b by Gabox", "[Roformer] MelBand Roformer | Instrumental FVX by Gabox", "[Roformer] MelBand Roformer | De-Reverb by anvuew", "[Roformer] MelBand Roformer | De-Reverb Less Aggressive by anvuew", "[Roformer] MelBand Roformer | De-Reverb Mono by anvuew", "[Roformer] MelBand Roformer | De-Reverb Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb Super Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo V2 by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo Fused by Sucial", "[Roformer] MelBand Roformer Kim | SYHFT by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2.5 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V3 by SYH99999", "[Roformer] MelBand Roformer Kim | Big SYHFT V1 by SYH99999", "[Roformer] MelBand Roformer Kim | Big Beta 4 FT by unwa", "[Roformer] MelBand Roformer Kim | Big Beta 5e FT by unwa", "[Roformer] MelBand Roformer | Big Beta 6 by unwa", "[Roformer] MelBand Roformer | Big Beta 6X by unwa", "[Roformer] BS Roformer | Vocals Revive by Unwa", "[Roformer] BS Roformer | Vocals Revive V2 by Unwa", "[Roformer] BS Roformer | Vocals Revive V3e by Unwa", "[Roformer] BS Roformer | Chorus Male-Female by Sucial", "[Roformer] BS Roformer | Male-Female by aufr33", "[Roformer] MelBand Roformer | Aspiration by Sucial", "[Roformer] MelBand Roformer | Aspiration Less Aggressive by Sucial", "[Roformer] MelBand Roformer | Bleed Suppressor V1 by unwa-97chris", "[Roformer] BS Roformer | Vocals Resurrection by unwa", "[Roformer] BS Roformer | Instrumental Resurrection by unwa", "[Roformer] BS Roformer | Instrumental ResurrectioN by Gabox", "[Roformer] BS Roformer SW by jarredou", "[MDX23C] MDX23C_D1581.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ_2.ckpt", "[MDX23C] MDX23C-De-Reverb-aufr33-jarredou.ckpt", "[MDX23C] MDX23C-DrumSep-aufr33-jarredou.ckpt", "[MDX-NET] UVR-MDX-NET-Inst_full_292.onnx", "[MDX-NET] UVR-MDX-NET_Inst_187_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_82_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_90_beta.onnx", "[MDX-NET] UVR-MDX-NET_Main_340.onnx", "[MDX-NET] UVR-MDX-NET_Main_390.onnx", "[MDX-NET] UVR-MDX-NET_Main_406.onnx", "[MDX-NET] UVR-MDX-NET_Main_427.onnx", "[MDX-NET] UVR-MDX-NET_Main_438.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_3.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_4.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_5.onnx", "[MDX-NET] UVR_MDXNET_Main.onnx", "[MDX-NET] UVR-MDX-NET-Inst_Main.onnx", "[MDX-NET] UVR_MDXNET_1_9703.onnx", "[MDX-NET] UVR_MDXNET_2_9682.onnx", "[MDX-NET] UVR_MDXNET_3_9662.onnx", "[MDX-NET] UVR-MDX-NET-Inst_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_3.onnx", "[MDX-NET] UVR_MDXNET_KARA.onnx", "[MDX-NET] UVR_MDXNET_KARA_2.onnx", "[MDX-NET] UVR_MDXNET_9482.onnx", "[MDX-NET] UVR-MDX-NET-Voc_FT.onnx", "[MDX-NET] Kim_Vocal_1.onnx", "[MDX-NET] Kim_Vocal_2.onnx", "[MDX-NET] Kim_Inst.onnx", "[MDX-NET] Reverb_HQ_By_FoxJoy.onnx", "[MDX-NET] UVR-MDX-NET_Crowd_HQ_1.onnx", "[MDX-NET] kuielab_a_vocals.onnx", "[MDX-NET] kuielab_a_other.onnx", "[MDX-NET] kuielab_a_bass.onnx", "[MDX-NET] kuielab_a_drums.onnx", "[MDX-NET] kuielab_b_vocals.onnx", "[MDX-NET] kuielab_b_other.onnx", "[MDX-NET] kuielab_b_bass.onnx", "[MDX-NET] kuielab_b_drums.onnx", "[VR Arch] 1_HP-UVR.pth", "[VR Arch] 2_HP-UVR.pth", "[VR Arch] 3_HP-Vocal-UVR.pth", "[VR Arch] 4_HP-Vocal-UVR.pth", "[VR Arch] 5_HP-Karaoke-UVR.pth", "[VR Arch] 6_HP-Karaoke-UVR.pth", "[VR Arch] 7_HP2-UVR.pth", "[VR Arch] 8_HP2-UVR.pth", "[VR Arch] 9_HP2-UVR.pth", "[VR Arch] 10_SP-UVR-2B-32000-1.pth", "[VR Arch] 11_SP-UVR-2B-32000-2.pth", "[VR Arch] 12_SP-UVR-3B-44100.pth", "[VR Arch] 13_SP-UVR-4B-44100-1.pth", "[VR Arch] 14_SP-UVR-4B-44100-2.pth", "[VR Arch] 15_SP-UVR-MID-44100-1.pth", "[VR Arch] 16_SP-UVR-MID-44100-2.pth", "[VR Arch] 17_HP-Wind_Inst-UVR.pth", "[VR Arch] UVR-De-Echo-Aggressive.pth", "[VR Arch] UVR-De-Echo-Normal.pth", "[VR Arch] UVR-DeEcho-DeReverb.pth", "[VR Arch] UVR-De-Reverb-aufr33-jarredou.pth", "[VR Arch] UVR-DeNoise-Lite.pth", "[VR Arch] UVR-DeNoise.pth", "[VR Arch] UVR-BVE-4B_SN-44100-1.pth", "[VR Arch] UVR-BVE-4B_SN-44100-2.pth", "[VR Arch] MGM_HIGHEND_v4.pth", "[VR Arch] MGM_LOWEND_A_v4.pth", "[VR Arch] MGM_LOWEND_B_v4.pth", "[VR Arch] MGM_MAIN_v4.pth", "[Demucs] htdemucs_ft.yaml", "[Demucs] htdemucs_6s.yaml", "[Demucs] htdemucs.yaml", "[Demucs] hdemucs_mmi.yaml"]
model_3 = "None" #@param ["None", "[Roformer] BS-Roformer-Viperx-1297", "[Roformer] BS-Roformer-Viperx-1296", "[Roformer] BS-Roformer-Viperx-1053", "[Roformer] Mel-Roformer-Viperx-1143", "[Roformer] BS-Roformer-De-Reverb", "[Roformer] Mel-Roformer-Crowd-Aufr33-Viperx", "[Roformer] Mel-Roformer-Denoise-Aufr33", "[Roformer] Mel-Roformer-Denoise-Aufr33-Aggr", "[Roformer] MelBand Roformer | Denoise-Debleed by Gabox", "[Roformer] Mel-Roformer-Karaoke-Aufr33-Viperx", "[Roformer] MelBand Roformer | Karaoke by Gabox", "[Roformer] MelBand Roformer | Karaoke V2 by Gabox", "[Roformer] MelBand Roformer | Karaoke by becruily", "[Roformer] BS Roformer | Karaoke by frazer-becruily", "[Roformer] BS Roformer | Karaoke by anvuew", "[Roformer] MelBand Roformer | Vocals by Kimberley Jensen", "[Roformer] MelBand Roformer Kim | FT by unwa", "[Roformer] MelBand Roformer Kim | FT 2 by unwa", "[Roformer] MelBand Roformer Kim | FT 2 Bleedless by unwa", "[Roformer] MelBand Roformer Kim | FT 3 by unwa", "[Roformer] MelBand Roformer Kim | Inst V1 by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V2 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V1 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V2 by Unwa", "[Roformer] MelBand Roformer | Vocals by becruily", "[Roformer] MelBand Roformer | Instrumental by becruily", "[Roformer] MelBand Roformer | Vocals Fullness by Aname", "[Roformer] BS Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals V2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV1 by Gabox", "[Roformer] MelBand Roformer | Vocals FV2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV3 by Gabox", "[Roformer] MelBand Roformer | Vocals FV4 by Gabox", "[Roformer] MelBand Roformer | Vocals FV5 by Gabox", "[Roformer] MelBand Roformer | Vocals FV6 by Gabox", "[Roformer] MelBand Roformer | Vocals FV7b by Gabox", "[Roformer] MelBand Roformer | Instrumental by Gabox", "[Roformer] MelBand Roformer | Instrumental 2 by Gabox", "[Roformer] MelBand Roformer | Instrumental 3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V4 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox", "[Roformer] MelBand Roformer | INSTV5 by Gabox", "[Roformer] MelBand Roformer | INSTV5N by Gabox", "[Roformer] MelBand Roformer | INSTV6 by Gabox", "[Roformer] MelBand Roformer | INSTV6N by Gabox", "[Roformer] MelBand Roformer | INSTV7 by Gabox", "[Roformer] MelBand Roformer | INSTV7N by Gabox", "[Roformer] MelBand Roformer | INSTV8 by Gabox", "[Roformer] MelBand Roformer | INSTV8N by Gabox", "[Roformer] MelBand Roformer | Instrumental FV7z by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8 by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8b by Gabox", "[Roformer] MelBand Roformer | Instrumental FVX by Gabox", "[Roformer] MelBand Roformer | De-Reverb by anvuew", "[Roformer] MelBand Roformer | De-Reverb Less Aggressive by anvuew", "[Roformer] MelBand Roformer | De-Reverb Mono by anvuew", "[Roformer] MelBand Roformer | De-Reverb Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb Super Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo V2 by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo Fused by Sucial", "[Roformer] MelBand Roformer Kim | SYHFT by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2.5 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V3 by SYH99999", "[Roformer] MelBand Roformer Kim | Big SYHFT V1 by SYH99999", "[Roformer] MelBand Roformer Kim | Big Beta 4 FT by unwa", "[Roformer] MelBand Roformer Kim | Big Beta 5e FT by unwa", "[Roformer] MelBand Roformer | Big Beta 6 by unwa", "[Roformer] MelBand Roformer | Big Beta 6X by unwa", "[Roformer] BS Roformer | Vocals Revive by Unwa", "[Roformer] BS Roformer | Vocals Revive V2 by Unwa", "[Roformer] BS Roformer | Vocals Revive V3e by Unwa", "[Roformer] BS Roformer | Chorus Male-Female by Sucial", "[Roformer] BS Roformer | Male-Female by aufr33", "[Roformer] MelBand Roformer | Aspiration by Sucial", "[Roformer] MelBand Roformer | Aspiration Less Aggressive by Sucial", "[Roformer] MelBand Roformer | Bleed Suppressor V1 by unwa-97chris", "[Roformer] BS Roformer | Vocals Resurrection by unwa", "[Roformer] BS Roformer | Instrumental Resurrection by unwa", "[Roformer] BS Roformer | Instrumental ResurrectioN by Gabox", "[Roformer] BS Roformer SW by jarredou", "[MDX23C] MDX23C_D1581.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ_2.ckpt", "[MDX23C] MDX23C-De-Reverb-aufr33-jarredou.ckpt", "[MDX23C] MDX23C-DrumSep-aufr33-jarredou.ckpt", "[MDX-NET] UVR-MDX-NET-Inst_full_292.onnx", "[MDX-NET] UVR-MDX-NET_Inst_187_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_82_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_90_beta.onnx", "[MDX-NET] UVR-MDX-NET_Main_340.onnx", "[MDX-NET] UVR-MDX-NET_Main_390.onnx", "[MDX-NET] UVR-MDX-NET_Main_406.onnx", "[MDX-NET] UVR-MDX-NET_Main_427.onnx", "[MDX-NET] UVR-MDX-NET_Main_438.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_3.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_4.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_5.onnx", "[MDX-NET] UVR_MDXNET_Main.onnx", "[MDX-NET] UVR-MDX-NET-Inst_Main.onnx", "[MDX-NET] UVR_MDXNET_1_9703.onnx", "[MDX-NET] UVR_MDXNET_2_9682.onnx", "[MDX-NET] UVR_MDXNET_3_9662.onnx", "[MDX-NET] UVR-MDX-NET-Inst_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_3.onnx", "[MDX-NET] UVR_MDXNET_KARA.onnx", "[MDX-NET] UVR_MDXNET_KARA_2.onnx", "[MDX-NET] UVR_MDXNET_9482.onnx", "[MDX-NET] UVR-MDX-NET-Voc_FT.onnx", "[MDX-NET] Kim_Vocal_1.onnx", "[MDX-NET] Kim_Vocal_2.onnx", "[MDX-NET] Kim_Inst.onnx", "[MDX-NET] Reverb_HQ_By_FoxJoy.onnx", "[MDX-NET] UVR-MDX-NET_Crowd_HQ_1.onnx", "[MDX-NET] kuielab_a_vocals.onnx", "[MDX-NET] kuielab_a_other.onnx", "[MDX-NET] kuielab_a_bass.onnx", "[MDX-NET] kuielab_a_drums.onnx", "[MDX-NET] kuielab_b_vocals.onnx", "[MDX-NET] kuielab_b_other.onnx", "[MDX-NET] kuielab_b_bass.onnx", "[MDX-NET] kuielab_b_drums.onnx", "[VR Arch] 1_HP-UVR.pth", "[VR Arch] 2_HP-UVR.pth", "[VR Arch] 3_HP-Vocal-UVR.pth", "[VR Arch] 4_HP-Vocal-UVR.pth", "[VR Arch] 5_HP-Karaoke-UVR.pth", "[VR Arch] 6_HP-Karaoke-UVR.pth", "[VR Arch] 7_HP2-UVR.pth", "[VR Arch] 8_HP2-UVR.pth", "[VR Arch] 9_HP2-UVR.pth", "[VR Arch] 10_SP-UVR-2B-32000-1.pth", "[VR Arch] 11_SP-UVR-2B-32000-2.pth", "[VR Arch] 12_SP-UVR-3B-44100.pth", "[VR Arch] 13_SP-UVR-4B-44100-1.pth", "[VR Arch] 14_SP-UVR-4B-44100-2.pth", "[VR Arch] 15_SP-UVR-MID-44100-1.pth", "[VR Arch] 16_SP-UVR-MID-44100-2.pth", "[VR Arch] 17_HP-Wind_Inst-UVR.pth", "[VR Arch] UVR-De-Echo-Aggressive.pth", "[VR Arch] UVR-De-Echo-Normal.pth", "[VR Arch] UVR-DeEcho-DeReverb.pth", "[VR Arch] UVR-De-Reverb-aufr33-jarredou.pth", "[VR Arch] UVR-DeNoise-Lite.pth", "[VR Arch] UVR-DeNoise.pth", "[VR Arch] UVR-BVE-4B_SN-44100-1.pth", "[VR Arch] UVR-BVE-4B_SN-44100-2.pth", "[VR Arch] MGM_HIGHEND_v4.pth", "[VR Arch] MGM_LOWEND_A_v4.pth", "[VR Arch] MGM_LOWEND_B_v4.pth", "[VR Arch] MGM_MAIN_v4.pth", "[Demucs] htdemucs_ft.yaml", "[Demucs] htdemucs_6s.yaml", "[Demucs] htdemucs.yaml", "[Demucs] hdemucs_mmi.yaml"]
model_4 = "None" #@param ["None", "[Roformer] BS-Roformer-Viperx-1297", "[Roformer] BS-Roformer-Viperx-1296", "[Roformer] BS-Roformer-Viperx-1053", "[Roformer] Mel-Roformer-Viperx-1143", "[Roformer] BS-Roformer-De-Reverb", "[Roformer] Mel-Roformer-Crowd-Aufr33-Viperx", "[Roformer] Mel-Roformer-Denoise-Aufr33", "[Roformer] Mel-Roformer-Denoise-Aufr33-Aggr", "[Roformer] MelBand Roformer | Denoise-Debleed by Gabox", "[Roformer] Mel-Roformer-Karaoke-Aufr33-Viperx", "[Roformer] MelBand Roformer | Karaoke by Gabox", "[Roformer] MelBand Roformer | Karaoke V2 by Gabox", "[Roformer] MelBand Roformer | Karaoke by becruily", "[Roformer] BS Roformer | Karaoke by frazer-becruily", "[Roformer] BS Roformer | Karaoke by anvuew", "[Roformer] MelBand Roformer | Vocals by Kimberley Jensen", "[Roformer] MelBand Roformer Kim | FT by unwa", "[Roformer] MelBand Roformer Kim | FT 2 by unwa", "[Roformer] MelBand Roformer Kim | FT 2 Bleedless by unwa", "[Roformer] MelBand Roformer Kim | FT 3 by unwa", "[Roformer] MelBand Roformer Kim | Inst V1 by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) by Unwa", "[Roformer] MelBand Roformer Kim | Inst V1 (E) Plus by Unwa", "[Roformer] MelBand Roformer Kim | Inst V2 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V1 by Unwa", "[Roformer] MelBand Roformer Kim | InstVoc Duality V2 by Unwa", "[Roformer] MelBand Roformer | Vocals by becruily", "[Roformer] MelBand Roformer | Instrumental by becruily", "[Roformer] MelBand Roformer | Vocals Fullness by Aname", "[Roformer] BS Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals by Gabox", "[Roformer] MelBand Roformer | Vocals V2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV1 by Gabox", "[Roformer] MelBand Roformer | Vocals FV2 by Gabox", "[Roformer] MelBand Roformer | Vocals FV3 by Gabox", "[Roformer] MelBand Roformer | Vocals FV4 by Gabox", "[Roformer] MelBand Roformer | Vocals FV5 by Gabox", "[Roformer] MelBand Roformer | Vocals FV6 by Gabox", "[Roformer] MelBand Roformer | Vocals FV7b by Gabox", "[Roformer] MelBand Roformer | Instrumental by Gabox", "[Roformer] MelBand Roformer | Instrumental 2 by Gabox", "[Roformer] MelBand Roformer | Instrumental 3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Bleedless V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V1 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V2 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V3 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness V4 by Gabox", "[Roformer] MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox", "[Roformer] MelBand Roformer | INSTV5 by Gabox", "[Roformer] MelBand Roformer | INSTV5N by Gabox", "[Roformer] MelBand Roformer | INSTV6 by Gabox", "[Roformer] MelBand Roformer | INSTV6N by Gabox", "[Roformer] MelBand Roformer | INSTV7 by Gabox", "[Roformer] MelBand Roformer | INSTV7N by Gabox", "[Roformer] MelBand Roformer | INSTV8 by Gabox", "[Roformer] MelBand Roformer | INSTV8N by Gabox", "[Roformer] MelBand Roformer | Instrumental FV7z by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8 by Gabox", "[Roformer] MelBand Roformer | Instrumental FV8b by Gabox", "[Roformer] MelBand Roformer | Instrumental FVX by Gabox", "[Roformer] MelBand Roformer | De-Reverb by anvuew", "[Roformer] MelBand Roformer | De-Reverb Less Aggressive by anvuew", "[Roformer] MelBand Roformer | De-Reverb Mono by anvuew", "[Roformer] MelBand Roformer | De-Reverb Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb Super Big by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo V2 by Sucial", "[Roformer] MelBand Roformer | De-Reverb-Echo Fused by Sucial", "[Roformer] MelBand Roformer Kim | SYHFT by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V2.5 by SYH99999", "[Roformer] MelBand Roformer Kim | SYHFT V3 by SYH99999", "[Roformer] MelBand Roformer Kim | Big SYHFT V1 by SYH99999", "[Roformer] MelBand Roformer Kim | Big Beta 4 FT by unwa", "[Roformer] MelBand Roformer Kim | Big Beta 5e FT by unwa", "[Roformer] MelBand Roformer | Big Beta 6 by unwa", "[Roformer] MelBand Roformer | Big Beta 6X by unwa", "[Roformer] BS Roformer | Vocals Revive by Unwa", "[Roformer] BS Roformer | Vocals Revive V2 by Unwa", "[Roformer] BS Roformer | Vocals Revive V3e by Unwa", "[Roformer] BS Roformer | Chorus Male-Female by Sucial", "[Roformer] BS Roformer | Male-Female by aufr33", "[Roformer] MelBand Roformer | Aspiration by Sucial", "[Roformer] MelBand Roformer | Aspiration Less Aggressive by Sucial", "[Roformer] MelBand Roformer | Bleed Suppressor V1 by unwa-97chris", "[Roformer] BS Roformer | Vocals Resurrection by unwa", "[Roformer] BS Roformer | Instrumental Resurrection by unwa", "[Roformer] BS Roformer | Instrumental ResurrectioN by Gabox", "[Roformer] BS Roformer SW by jarredou", "[MDX23C] MDX23C_D1581.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ.ckpt", "[MDX23C] MDX23C-8KFFT-InstVoc_HQ_2.ckpt", "[MDX23C] MDX23C-De-Reverb-aufr33-jarredou.ckpt", "[MDX23C] MDX23C-DrumSep-aufr33-jarredou.ckpt", "[MDX-NET] UVR-MDX-NET-Inst_full_292.onnx", "[MDX-NET] UVR-MDX-NET_Inst_187_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_82_beta.onnx", "[MDX-NET] UVR-MDX-NET_Inst_90_beta.onnx", "[MDX-NET] UVR-MDX-NET_Main_340.onnx", "[MDX-NET] UVR-MDX-NET_Main_390.onnx", "[MDX-NET] UVR-MDX-NET_Main_406.onnx", "[MDX-NET] UVR-MDX-NET_Main_427.onnx", "[MDX-NET] UVR-MDX-NET_Main_438.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_3.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_4.onnx", "[MDX-NET] UVR-MDX-NET-Inst_HQ_5.onnx", "[MDX-NET] UVR_MDXNET_Main.onnx", "[MDX-NET] UVR-MDX-NET-Inst_Main.onnx", "[MDX-NET] UVR_MDXNET_1_9703.onnx", "[MDX-NET] UVR_MDXNET_2_9682.onnx", "[MDX-NET] UVR_MDXNET_3_9662.onnx", "[MDX-NET] UVR-MDX-NET-Inst_1.onnx", "[MDX-NET] UVR-MDX-NET-Inst_2.onnx", "[MDX-NET] UVR-MDX-NET-Inst_3.onnx", "[MDX-NET] UVR_MDXNET_KARA.onnx", "[MDX-NET] UVR_MDXNET_KARA_2.onnx", "[MDX-NET] UVR_MDXNET_9482.onnx", "[MDX-NET] UVR-MDX-NET-Voc_FT.onnx", "[MDX-NET] Kim_Vocal_1.onnx", "[MDX-NET] Kim_Vocal_2.onnx", "[MDX-NET] Kim_Inst.onnx", "[MDX-NET] Reverb_HQ_By_FoxJoy.onnx", "[MDX-NET] UVR-MDX-NET_Crowd_HQ_1.onnx", "[MDX-NET] kuielab_a_vocals.onnx", "[MDX-NET] kuielab_a_other.onnx", "[MDX-NET] kuielab_a_bass.onnx", "[MDX-NET] kuielab_a_drums.onnx", "[MDX-NET] kuielab_b_vocals.onnx", "[MDX-NET] kuielab_b_other.onnx", "[MDX-NET] kuielab_b_bass.onnx", "[MDX-NET] kuielab_b_drums.onnx", "[VR Arch] 1_HP-UVR.pth", "[VR Arch] 2_HP-UVR.pth", "[VR Arch] 3_HP-Vocal-UVR.pth", "[VR Arch] 4_HP-Vocal-UVR.pth", "[VR Arch] 5_HP-Karaoke-UVR.pth", "[VR Arch] 6_HP-Karaoke-UVR.pth", "[VR Arch] 7_HP2-UVR.pth", "[VR Arch] 8_HP2-UVR.pth", "[VR Arch] 9_HP2-UVR.pth", "[VR Arch] 10_SP-UVR-2B-32000-1.pth", "[VR Arch] 11_SP-UVR-2B-32000-2.pth", "[VR Arch] 12_SP-UVR-3B-44100.pth", "[VR Arch] 13_SP-UVR-4B-44100-1.pth", "[VR Arch] 14_SP-UVR-4B-44100-2.pth", "[VR Arch] 15_SP-UVR-MID-44100-1.pth", "[VR Arch] 16_SP-UVR-MID-44100-2.pth", "[VR Arch] 17_HP-Wind_Inst-UVR.pth", "[VR Arch] UVR-De-Echo-Aggressive.pth", "[VR Arch] UVR-De-Echo-Normal.pth", "[VR Arch] UVR-DeEcho-DeReverb.pth", "[VR Arch] UVR-De-Reverb-aufr33-jarredou.pth", "[VR Arch] UVR-DeNoise-Lite.pth", "[VR Arch] UVR-DeNoise.pth", "[VR Arch] UVR-BVE-4B_SN-44100-1.pth", "[VR Arch] UVR-BVE-4B_SN-44100-2.pth", "[VR Arch] MGM_HIGHEND_v4.pth", "[VR Arch] MGM_LOWEND_A_v4.pth", "[VR Arch] MGM_LOWEND_B_v4.pth", "[VR Arch] MGM_MAIN_v4.pth", "[Demucs] htdemucs_ft.yaml", "[Demucs] htdemucs_6s.yaml", "[Demucs] htdemucs.yaml", "[Demucs] hdemucs_mmi.yaml"]
algorithm = "avg_wave" #@param ["avg_wave", "median_wave", "min_wave", "max_wave", "avg_fft", "median_fft", "min_fft", "max_fft", "uvr_max_spec", "uvr_min_spec", "ensemble_wav"]
#@markdown Weight of each model, separated by commas (e.g. `2, 1`). Only used by avg_wave and avg_fft; empty = equal weights.
weights = "" #@param {type:"string"}

#@markdown Select the output format:
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]
#@markdown Also output the stems that only some models produce.
include_unmatched = False #@param {type:"boolean"}
#@markdown Keep the stems of each model in a `<song>_ensemble_parts` folder.
keep_intermediates = False #@param {type:"boolean"}
#@markdown ### **Output levels**
#@markdown Max peak of the output: louder outputs are scaled down to it.
normalization_threshold = 0.9 #@param {type:"slider", min:0.1, max:1, step:0.1}
#@markdown Min peak of the output: quieter outputs are scaled up to it.
amplification_threshold = 0.7 #@param {type:"slider", min:0.1, max:1, step:0.1}

run_ensemble(preset, [model_1, model_2, model_3, model_4], algorithm, weights, audio_input, output_folder,
             output_format, include_unmatched, keep_intermediates, normalization_threshold, amplification_threshold)

# <small> **Extra**
Base code by [Blane187](https://github.com/Blane187)

In [ ]:
#@markdown #**Audio Downloader for Separation**
#@markdown Link:
video_url = "https://youtu.be/L3P0LgwuPPo?si=8uQWCX9kiXgMxYdE" #@param {type:"string"}
#@markdown You can paste the link to the video/audio from many sites, check the complete list [here](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown Audio name (empty = use the video title):
audio_name = "" #@param {type:"string"}
#@markdown Folder for the downloaded audio file:
save_folder = "/content/drive/MyDrive/UVR5/inputs" #@param {type:"string"}
#@markdown Select the output format:
audio_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a"]

run_download(video_url, save_folder, audio_format, audio_name)